# DCASE 2026 Task 2 — PART 1 MASTER NOTEBOOK: all seven machine types

**Project:** Noise-aware Unsupervised Anomalous Sound Detection for Machine Condition Monitoring.
**This notebook (Part 1):** for each of the seven machine types: archive check → selective extraction → full dataset audit → two-channel log-Mel features → Experiment 1 / Experiment 3 training (Model A source-only, Model B source + target) → Experiment 2 preparation (latent and residual Mahalanobis statistics) → verification → persistent results on Google Drive → one tested handover ZIP.
**Not here (Part 2, teammate):** anomaly scores, thresholds, AUC, pAUC, precision, recall, F1, result comparison, application.

## Before you run: transfer the archives from Windows to Google Drive
1. On your Windows computer open https://drive.google.com in a browser and sign in with the same Google account you use for Colab.
2. Open the folder `My Drive → DCASE_2026 → Dataset` (it already contains `dev_ToyCar_r2.zip`).
3. Drag the six ZIP files from your Downloads folder into that Drive folder **with their original names, unchanged**:
   `dev_ToyCarEmu.zip`, `dev_fan.zip`, `dev_gearboxEmu.zip`, `dev_bearingEmu.zip`, `dev_sliderEmu.zip`, `dev_valveEmu.zip` (about 4.1 GB in total; on a typical home connection 30 to 90 minutes). The browser shows an upload progress panel; keep the tab open until every file says "Uploaded". Alternatively use the Google Drive for Desktop app and copy the files into the synced `DCASE_2026\Dataset` folder.
4. Free Drive space needed: about 4.8 GB for the archives plus about 1 GB for results. If your Drive is almost full, upload three archives, run the notebook, then upload the rest: the notebook processes whatever archives are present and resumes later with the others.
5. Do **not** extract the ZIPs on Drive; the notebook extracts each one on Colab's local disk, one machine at a time, and deletes the local copy afterwards. The ZIPs themselves are never modified or deleted.

## How the run works
1. Runtime → Change runtime type → **T4 GPU** (any GPU is fine; CPU also works but about ten times slower). Then **Runtime → Run all**.
2. **Stage 1** checks the seven dataset archives on Drive and prints their internal structure (no extraction yet).
3. **Stage 2** checks the compatibility of the earlier ToyCar models in `DCASE_2026_PART1_RESULTS/` and preserves them as reference models (`Models/ToyCar/Previous_Holdout_Validation_Run/`). Under the approved `spec_full` protocol they are **incompatible** (they were trained with a 99-recording validation hold-out on 891 / 891 + 10 recordings) and are **not** reused as primary deliverables. The original ToyCar results folder is never overwritten.
4. **Stage 3** processes **all seven machines, including ToyCar**, strictly one at a time. New ToyCar Model A and Model B are trained on 990 source and 990 source + 10 target recordings respectively, exactly like the other six machines. Each machine takes roughly 35 to 50 minutes on a T4, so about 4 to 6 hours in total.
5. Colab will probably disconnect at least once. **Just reconnect and Run all again.** Stage 1 and Stage 2 repeat their checks (the reference copy is made only once); in Stage 3 completed machines are skipped (`already COMPLETE, skipping`), a finished model inside an unfinished machine is skipped (`already COMPLETE (selected epoch ...)`), a half-trained model continues from its last saved epoch (`RESUMING from epoch N (spec_full: no validation set)`), Mahalanobis statistics and verification are recomputed only where their markers are missing, and the current machine's archive is re-extracted automatically from Drive. Nothing finished is ever redone unless you set `OVERWRITE_COMPLETED = True`. Do not change CONFIG between runs (only `epochs` may change; other changes are refused by the resume guard).
6. When Stage 5 prints `HANDOVER ZIP VERIFIED` and Stage 6 prints the final status, send `My Drive/DCASE_2026_PART1_COMPLETE/DCASE_2026_PART1_COMPLETE_HANDOVER.zip` to your teammate.

## Methodology (features and architecture identical to the earlier ToyCar run; split protocol `spec_full` as approved; deviations from the official baseline are documented in the report)
| Item | Value |
|---|---|
| Input | both channels (Channel 1, Channel 2: neutral labels), 128 log-Mel each, n_fft 1024, hop 512, power 2, 5 frames → **1280-dim** |
| Normalisation | per-dimension z-score fitted on each model's own training recordings |
| Model | 1280 → 128 → 128 → 128 → 128 → 8 → 128 → 128 → 128 → 128 → 1280, BatchNorm + ReLU, 432 408 parameters |
| Training | Adam 1e-3, batch 256, 100 epochs, MSE, seed 42 (early stopping only under the alternative `holdout_10pct` protocol) |
| Split (`SPLIT_PROTOCOL = "spec_full"`, approved) | Model A: all 990 source normals; Model B: all 990 source + 10 target normals; fixed 100 epochs; final-epoch checkpoint; **no held-out validation set, no early stopping, no validation loss**; test clips never used |
| Experiment 2 preparation | mean, covariance, regularised precision of the 8-dim latent and of the 256-dim residual frames, per model and per training domain |

## Section 1 — Environment (Colab, Drive, hardware)
Expected output: `Running in Google Colab: True`, Drive mounted, library versions, GPU name and memory (or CPU notice), RAM and local disk figures.

In [ ]:
# ---- 1.1 Colab detection, dependency check ----------------------------------------------------
import sys, platform, importlib, importlib.util, subprocess
print("Python :", sys.version.split()[0], "|", platform.system(), platform.release())
assert sys.version_info >= (3, 9), "Python 3.9 or newer is required."
IN_COLAB = importlib.util.find_spec("google.colab") is not None
print("Running in Google Colab:", IN_COLAB)
REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib",
            "sklearn": "scikit-learn", "librosa": "librosa", "soundfile": "soundfile", "torch": "torch"}
missing = [pipname for mod, pipname in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing); importlib.invalidate_caches()
else:
    print("All required packages are already installed.")

In [ ]:
# ---- 1.2 Imports --------------------------------------------------------------------------------
import os, json, time, math, shutil, random, hashlib, warnings, zipfile, datetime, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import scipy, sklearn, matplotlib
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, BatchSampler, RandomSampler, SequentialSampler
from IPython.display import display
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
VERSIONS = {m: importlib.import_module(m).__version__ for m in REQUIRED}
display(pd.DataFrame.from_dict(VERSIONS, orient="index", columns=["version"]))

In [ ]:
# ---- 1.3 Mount Google Drive -----------------------------------------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive"); assert DRIVE_ROOT.exists(), "Drive did not mount; re-run and accept the dialog."
    print("Google Drive mounted:", DRIVE_ROOT)
else:
    DRIVE_ROOT = Path.cwd(); print("Not in Colab: using the current folder instead of Drive ->", DRIVE_ROOT)

In [ ]:
# ============================ CONFIG: the only cell you may need to edit ============================
DRIVE_DATA_DIR = DRIVE_ROOT / "DCASE_2026" / "Dataset"            # the seven ZIP archives live here (never modified)
ARCHIVES = {                                                      # canonical machine name -> exact ZIP filename on Drive
    "ToyCar":       "dev_ToyCar_r2.zip",
    "ToyCarEmu":    "dev_ToyCarEmu.zip",
    "Fan":          "dev_fan.zip",
    "GearboxEmu":   "dev_gearboxEmu.zip",
    "BearingEmu":   "dev_bearingEmu.zip",
    "SlideRailEmu": "dev_sliderEmu.zip",
    "ValveEmu":     "dev_valveEmu.zip",
}
MACHINE_ORDER = list(ARCHIVES)                                    # processed strictly one after the other
EXISTING_TOYCAR_RESULTS = DRIVE_ROOT / "DCASE_2026_PART1_RESULTS" # finished ToyCar run (full-training tree); reused, never overwritten
PROJECT_ROOT   = DRIVE_ROOT / "DCASE_2026_PART1_COMPLETE"         # final project directory (persistent)
LOCAL_WORK     = Path("/content/dcase_work") if IN_COLAB else Path.cwd() / "dcase_work"   # temporary: extraction + feature cache

SMOKE_TEST          = False   # True = tiny pipeline check (12 recordings per machine, 2 epochs) into PROJECT_ROOT + "_SMOKE"
RESUME              = True    # continue interrupted experiments from their last saved epoch
OVERWRITE_COMPLETED = False   # True = explicitly retrain finished experiments (old files archived first). Never affects ToyCar reuse.
STRICT_COUNTS       = True    # a machine whose archive does not contain 990/10 training and 200 test clips is marked INCOMPLETE and skipped
DELETE_LOCAL_AFTER_MACHINE = True   # free Colab disk after a machine is complete and verified (Drive outputs stay)
FEATURE_CACHE_ON_DRIVE     = False  # True = keep per-machine feature caches on Drive (~0.4 GB each); False = local only (rebuilt in ~5 min after a disconnect)

# ---- Experiment 3 split protocol (see Section 0 of the README) -----------------------------------
#   "spec_full"     : AI_LAB.pdf protocol. Model A = ALL 990 source normals, Model B = ALL 990 source + 10 target normals used for
#                     gradient updates; fixed 100 epochs; the final-epoch checkpoint is the model (no validation set, like the official
#                     baseline). Nothing is held out, so no recording-level validation loss exists; training loss is monitored per epoch.
#   "holdout_10pct" : the protocol of the finished ToyCar run: 99 source normals held out as a shared validation set, early stopping,
#                     best checkpoint; Model A trains on 891, Model B on 891 + 10.
SPLIT_PROTOCOL = "spec_full"

COMMON_CONFIG = dict(      # features and architecture identical to the finished ToyCar run
    sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128, context_frames=5, power=2.0,
    channel_mode="concat", normalise=True, window_stride=1,
    epochs=100, batch_size=256, learning_rate=1e-3,
    snapshot_every=10, seed=42, num_workers=0,
)
if SPLIT_PROTOCOL == "spec_full":
    COMMON_CONFIG.update(val_fraction=0.0, early_stop_patience=None, split_protocol="spec_full")
elif SPLIT_PROTOCOL == "holdout_10pct":
    COMMON_CONFIG.update(val_fraction=0.10, early_stop_patience=15, split_protocol="holdout_10pct")
else:
    raise SystemExit("SPLIT_PROTOCOL must be 'spec_full' or 'holdout_10pct'")
# ---- Optional: cross-fitted calibration scores for Part 2 threshold selection (no test data involved) -------------------
# CALIBRATION_FOLDS = 0 (default) -> off.  K >= 2 -> per machine, K extra "fold models" are trained on (K-1)/K of the SOURCE normals and
# score the held-out fold; the pooled out-of-fold scores of all 990 source normals are genuinely independent normal scores
# (Calibration/<m>_out_of_fold_normal_scores.csv). Extra cost per machine ~ K * CALIBRATION_EPOCHS/100 of one model's training time
# (K=5, 30 epochs: about 1.5 model-trainings, roughly 20 minutes on a T4; about 2 to 3 hours on CPU). Fold models never see test data.
CALIBRATION_FOLDS  = 0
CALIBRATION_EPOCHS = 30

EXPECTED = dict(train_source_normal=990, train_target_normal=10, test_source=100, test_target=100, total=1200)
SMOKE_FILES = 12
if SMOKE_TEST:
    COMMON_CONFIG.update(epochs=2, early_stop_patience=None, snapshot_every=1)
    PROJECT_ROOT = PROJECT_ROOT.parent / (PROJECT_ROOT.name + "_SMOKE")

In [ ]:
# ---- 1.4 Hardware, seeds, project folders, log ---------------------------------------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(COMMON_CONFIG["seed"])

if torch.cuda.is_available():
    DEVICE = torch.device("cuda"); _p = torch.cuda.get_device_properties(0)
    GPU_GB = _p.total_memory / 1024**3
    print(f"GPU: {_p.name} | {GPU_GB:.1f} GB | CUDA {torch.version.cuda}")
else:
    DEVICE = torch.device("cpu"); GPU_GB = 0.0
    torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))
    print(f"No GPU -> CPU with {torch.get_num_threads()} threads (works, roughly 10x slower).")
INFER_BATCH = 8192 if GPU_GB >= 12 else 4096 if GPU_GB >= 6 else 1024    # inference batch adapts to memory; training batch stays 256 for comparability
def mem_info():
    try:
        tot = {l.split()[0]: int(l.split()[1]) / 1024**2 for l in open("/proc/meminfo") if l.split()[0] in ("MemTotal:", "MemAvailable:")}
        return f"RAM total {tot.get('MemTotal:', 0):.1f} GB, available {tot.get('MemAvailable:', 0):.1f} GB"
    except Exception: return "RAM: n/a"
def disk_info(p: Path):
    u = shutil.disk_usage(str(p if p.exists() else p.parent)); return f"disk at {p}: free {u.free/1024**3:.1f} GB of {u.total/1024**3:.1f} GB"
print(mem_info()); print(disk_info(LOCAL_WORK.parent)); print(disk_info(DRIVE_ROOT))

D = dict(
    calib=PROJECT_ROOT / "Calibration",
    audit=PROJECT_ROOT / "Dataset_Audit", pre_code=PROJECT_ROOT / "Preprocessing" / "preprocessing_code", pre_cfg=PROJECT_ROOT / "Preprocessing" / "preprocessing_configs",
    models=PROJECT_ROOT / "Models", hist=PROJECT_ROOT / "Training_Histories", curves=PROJECT_ROOT / "Loss_Curves", mahala=PROJECT_ROOT / "Mahalanobis_Statistics",
    cfg=PROJECT_ROOT / "Configurations", verify=PROJECT_ROOT / "Verification_Reports", notebooks=PROJECT_ROOT / "Notebooks", handover=PROJECT_ROOT / "Handover", logs=PROJECT_ROOT / "logs",
)
for p in D.values(): p.mkdir(parents=True, exist_ok=True)
(LOCAL_WORK / "extract").mkdir(parents=True, exist_ok=True); (LOCAL_WORK / "cache").mkdir(parents=True, exist_ok=True)
LOG_FILE = D["logs"] / ("part1_master_smoke.log" if SMOKE_TEST else "part1_master.log")
def log(msg: str):
    line = f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"; print(msg, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f: f.write(line + "\n")
log(f"=== MASTER session start | mode={'SMOKE' if SMOKE_TEST else 'FULL'} | device={DEVICE} | project={PROJECT_ROOT} ===")
(D["cfg"] / "common_config.json").write_text(json.dumps(COMMON_CONFIG, indent=2))
(D["cfg"] / "environment.json").write_text(json.dumps(dict(in_colab=IN_COLAB, device=str(DEVICE), gpu_gb=GPU_GB, versions=VERSIONS, python=sys.version, started=datetime.datetime.now().isoformat()), indent=2))

STATE_FILE = D["cfg"] / "pipeline_state.json"
def load_state() -> dict:
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {m: {"status": "pending", "steps": {}, "notes": []} for m in MACHINE_ORDER}
def save_state(state: dict): STATE_FILE.write_text(json.dumps(state, indent=2, default=str))
def mark(state: dict, machine: str, step: str, value=True, note: str | None = None, status: str | None = None):
    state.setdefault(machine, {"status": "pending", "steps": {}, "notes": []})
    state[machine]["steps"][step] = dict(value=value, at=datetime.datetime.now().isoformat())
    if note: state[machine]["notes"].append(note)
    if status: state[machine]["status"] = status
    save_state(state)
STATE = load_state(); save_state(STATE)
print("\nPipeline state:", {m: STATE[m]["status"] for m in MACHINE_ORDER})

## Section 2 — Library: preprocessing, normalisation, model, inference
These four `[LIB]` cells are byte-identical to the ToyCar notebook, so feature extraction, normalisation, architecture and inference are the same for all seven machines and for the earlier ToyCar run. (Whether the earlier ToyCar checkpoints can be reused is a separate question of the split protocol, decided in Stage 2; under `spec_full` they are reference models only.) The cells are also written out as `Preprocessing/preprocessing_code/dcase_part1_lib.py` for Part 2.

In [ ]:
# [LIB] 3.1 Loading and preprocessing functions (shipped to Part 2 unchanged)
EPS = sys.float_info.epsilon
CHANNEL_LABELS = ["Channel 1", "Channel 2"]   # neutral: the file metadata does not identify near/far microphones

def load_audio(path, target_sr: int):
    # returns float32 array (samples, channels) at target_sr, and the file's original sample rate
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if sr != target_sr:
        audio = np.stack([librosa.resample(audio[:, c], orig_sr=sr, target_sr=target_sr) for c in range(audio.shape[1])], axis=1)
    return audio.astype(np.float32), sr

def select_channels(audio: np.ndarray, mode: str) -> np.ndarray:
    # (samples, 2) for "concat"; (samples, 1) for "mean" and "ch0"
    if audio.shape[1] == 1:
        if mode == "concat":
            raise ValueError("channel_mode='concat' needs 2-channel audio but this file is mono.")
        return audio
    if mode == "concat": return audio[:, :2]
    if mode == "mean":   return audio[:, :2].mean(axis=1, keepdims=True)
    if mode == "ch0":    return audio[:, :1]
    raise ValueError(f"unknown channel_mode {mode}")

def log_mel_per_channel(audio_1d: np.ndarray, cfg: dict) -> np.ndarray:
    mel = librosa.feature.melspectrogram(y=audio_1d, sr=cfg["sample_rate"], n_fft=cfg["n_fft"],
                                         hop_length=cfg["hop_length"], n_mels=cfg["n_mels"], power=cfg["power"])
    log_mel = (20.0 / cfg["power"]) * np.log10(np.maximum(mel, EPS))   # official baseline formula
    return log_mel.T.astype(np.float32)                                  # (frames, n_mels)

def wav_to_frames(path, cfg: dict) -> np.ndarray:
    # ONE function for training AND inference: WAV -> (frames, n_mels * channels_used), un-normalised dB
    audio, _ = load_audio(path, cfg["sample_rate"])
    chans = select_channels(audio, cfg["channel_mode"])
    frames = np.concatenate([log_mel_per_channel(chans[:, c], cfg) for c in range(chans.shape[1])], axis=1)
    frames = np.nan_to_num(frames, nan=0.0, posinf=0.0, neginf=-200.0)
    assert np.isfinite(frames).all(), "non-finite values after preprocessing"
    if frames.shape[0] < cfg["context_frames"]:
        raise ValueError(f"{Path(path).name}: only {frames.shape[0]} frames, need >= {cfg['context_frames']}")
    return frames

In [ ]:
# [LIB] 4.4 Normalisation (fit on training frames only; apply everywhere)
def fit_norm_stats(frames_list: list) -> dict:
    allf = np.concatenate(frames_list, axis=0)
    mean, std = allf.mean(axis=0), allf.std(axis=0)
    std = np.where(std < 1e-5, 1.0, std)
    return dict(mean=mean.astype(np.float32), std=std.astype(np.float32))

def apply_norm(frames: np.ndarray, stats: dict, cfg: dict) -> np.ndarray:
    return ((frames - stats["mean"]) / stats["std"]).astype(np.float32) if cfg["normalise"] else np.asarray(frames, np.float32)

In [ ]:
# [LIB] 5.1 Model definition
def dense_block(i: int, o: int) -> nn.Sequential:
    return nn.Sequential(nn.Linear(i, o), nn.BatchNorm1d(o, momentum=0.01, eps=1e-3), nn.ReLU())

class DCASEAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 128, latent: int = 8, n_hidden: int = 4):
        super().__init__()
        self.input_dim, self.latent_dim = input_dim, latent
        enc = [dense_block(input_dim, hidden)] + [dense_block(hidden, hidden) for _ in range(n_hidden - 1)] + [dense_block(hidden, latent)]
        dec = [dense_block(latent, hidden)] + [dense_block(hidden, hidden) for _ in range(n_hidden - 1)] + [nn.Linear(hidden, input_dim)]
        self.encoder, self.decoder = nn.Sequential(*enc), nn.Sequential(*dec)
    def encode(self, x): return self.encoder(x)
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def count_params(m: nn.Module) -> int: return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# [LIB] 9.1 Loading a checkpoint and running inference (shipped to Part 2 unchanged)
def load_checkpoint(path):
    # returns (model in eval mode on CPU, checkpoint dict); architecture is rebuilt from the stored input_dim
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model = DCASEAutoencoder(ck["input_dim"]); model.load_state_dict(ck["model_state_dict"]); model.eval()
    return model, ck

def checkpoint_norm_stats(ck: dict) -> dict:
    return dict(mean=np.asarray(ck["norm_mean"], np.float32), std=np.asarray(ck["norm_std"], np.float32))

def wav_to_model_input(path, ck: dict) -> np.ndarray:
    # exact training-time representation: (n_windows, input_dim) float32, normalised with the checkpoint's own statistics
    cfg = ck["config"]
    frames = apply_norm(wav_to_frames(path, cfg), checkpoint_norm_stats(ck), cfg)
    n = frames.shape[0] - cfg["context_frames"] + 1
    idx = np.arange(n)[:, None] + np.arange(cfg["context_frames"])[None, :]
    return frames[idx].reshape(n, -1).astype(np.float32)

@torch.no_grad()
def run_inference(model, x: np.ndarray, batch: int = 4096) -> dict:
    # returns reconstruction (n, input_dim), latent (n, 8), window_mse (n,), residual = input - reconstruction (n, input_dim)
    xt = torch.from_numpy(x); recs, lats = [], []
    for s in range(0, len(xt), batch):
        r, z = model(xt[s:s + batch]); recs.append(r); lats.append(z)
    rec, lat = torch.cat(recs).numpy(), torch.cat(lats).numpy(); resid = x - rec
    return dict(reconstruction=rec, latent=lat, window_mse=(resid ** 2).mean(axis=1), residual=resid)

def wav_to_everything(wav_path, checkpoint_path) -> dict:
    # Convenience for Part 2: WAV + checkpoint -> input, reconstruction, latent, window_mse, residual, config
    model, ck = load_checkpoint(checkpoint_path)
    x = wav_to_model_input(wav_path, ck)
    out = run_inference(model, x); out["input"] = x; out["config"] = ck["config"]; out["frame_dim"] = ck["frame_dim"]
    return out

In [ ]:
# ---- 2.5 Dataset helpers, windows, DataLoader (shared by all machines) ----------------------------
FRAME_DIM = COMMON_CONFIG["n_mels"] * (2 if COMMON_CONFIG["channel_mode"] == "concat" else 1)
INPUT_DIM = FRAME_DIM * COMMON_CONFIG["context_frames"]
LOSS_FN = nn.MSELoss()
print(f"frame dim {FRAME_DIM} | model input dim {INPUT_DIM}")

def parse_dcase_name(name: str) -> dict:
    parts = Path(name).stem.split("_")
    info = dict(filename=Path(name).name, section=None, domain=None, split=None, label=None, index=None, attributes=None)
    if len(parts) >= 6 and parts[0] == "section":
        info.update(section=parts[1], domain=parts[2], split=parts[3], label=parts[4], index=parts[5], attributes="_".join(parts[6:]) or None)
    return info

def window_starts(offsets, file_idx, context, stride):
    starts = [np.arange(offsets[i], offsets[i + 1] - context + 1, stride, dtype=np.int64) for i in file_idx]
    return np.concatenate(starts) if starts else np.zeros(0, np.int64)

class WindowDataset(Dataset):
    def __init__(self, frames: torch.Tensor, starts: np.ndarray, context: int):
        self.frames, self.starts, self.context, self.arange = frames, torch.as_tensor(starts), context, torch.arange(context)
    def __len__(self): return len(self.starts)
    def __getitem__(self, idx):
        pos = self.starts[torch.as_tensor(idx)].unsqueeze(-1) + self.arange
        return self.frames[pos].reshape(pos.shape[0], -1)

def make_loader(ds, batch_size, shuffle, drop_last, seed):
    g = torch.Generator(); g.manual_seed(seed)
    sampler = RandomSampler(ds, generator=g) if shuffle else SequentialSampler(ds)
    return DataLoader(ds, batch_size=None, sampler=BatchSampler(sampler, batch_size, drop_last), num_workers=COMMON_CONFIG["num_workers"])

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## Section 3 — Pipeline functions (one machine at a time)
Each function below is one step of the per-machine pipeline. They are defined here and called from the loop in Section 6. Every step writes its outputs to Drive and a marker, so an interrupted run resumes at the step that was in progress.

In [ ]:
# ---- 3.1 Archive inspection (ZIP index only, no extraction) and selective extraction ----------------
def inspect_archive(zip_path: Path) -> dict:
    # Reads the ZIP table of contents and derives the structure: machine folder, train/test/supplemental counts, attribute csv.
    info = dict(zip=zip_path.name, exists=zip_path.exists())
    if not zip_path.exists(): return info
    info["size_mb"] = round(zip_path.stat().st_size / 1e6, 1)
    with zipfile.ZipFile(zip_path) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
    wavs = [n for n in names if n.lower().endswith(".wav")]
    train = [n for n in wavs if "/train/" in n]; test = [n for n in wavs if "/test/" in n]; supp = [n for n in wavs if "/supplemental/" in n]
    other = [n for n in wavs if n not in set(train) | set(test) | set(supp)]
    machine_dirs = sorted({n.split("/train/")[0] for n in train})
    csvs = [n for n in names if n.lower().endswith(".csv")]
    def cnt(lst, dom=None, lab=None):
        return sum(1 for n in lst if (dom is None or parse_dcase_name(n)["domain"] == dom) and (lab is None or parse_dcase_name(n)["label"] == lab))
    dup_names = pd.Series([Path(n).name for n in train + test]).duplicated().sum()
    info.update(total_files=len(names), wav_files=len(wavs), machine_dir_in_zip=machine_dirs[0] if len(machine_dirs) == 1 else machine_dirs,
                train_total=len(train), train_source_normal=cnt(train, "source", "normal"), train_target_normal=cnt(train, "target", "normal"),
                train_other=len(train) - cnt(train, "source", "normal") - cnt(train, "target", "normal"),
                test_total=len(test), test_source=cnt(test, "source"), test_target=cnt(test, "target"),
                test_source_normal=cnt(test, "source", "normal"), test_source_anomaly=cnt(test, "source", "anomaly"),
                test_target_normal=cnt(test, "target", "normal"), test_target_anomaly=cnt(test, "target", "anomaly"),
                supplemental_wavs=len(supp), other_wavs=len(other), duplicate_filenames=int(dup_names), attribute_csvs=csvs,
                top_level_entries=sorted({n.split("/")[0] for n in names})[:10])
    return info

def extract_machine(zip_path: Path, dest: Path) -> Path:
    # Extracts ONLY train/, test/ and csv files (supplemental noise clips are not needed for Part 1 and are skipped).
    marker = dest / ".extraction_complete"
    if marker.exists():
        log(f"  local extraction already present: {dest}"); return dest
    if dest.exists(): shutil.rmtree(dest)
    dest.mkdir(parents=True)
    t0 = time.time()
    with zipfile.ZipFile(zip_path) as z:
        bad = z.testzip()
        if bad is not None: raise RuntimeError(f"{zip_path.name} is corrupted (first bad member {bad}); re-upload it to Drive.")
        members = [n for n in z.namelist() if (("/train/" in n or "/test/" in n) and n.lower().endswith(".wav")) or n.lower().endswith(".csv")]
        for i, n in enumerate(members, 1):
            z.extract(n, dest)
            if i % 400 == 0: log(f"  extracted {i}/{len(members)} files ({time.time() - t0:.0f}s)")
    marker.write_text(datetime.datetime.now().isoformat())
    log(f"  extraction of {len(members)} files finished in {time.time() - t0:.0f}s -> {dest}")
    return dest

def find_machine_dir(root: Path) -> Path:
    cands = sorted({p.parent for p in root.rglob("train") if p.is_dir() and any(p.glob("*.wav"))})
    if len(cands) != 1: raise RuntimeError(f"expected exactly one machine folder with train/*.wav under {root}, found {cands}")
    return cands[0]

def list_wavs(mdir: Path, split: str) -> list:
    d = mdir / split; return sorted(d.glob("*.wav")) if d.is_dir() else []

In [ ]:
# ---- 3.2 Full audit of an extracted machine (every file header, md5 for duplicate contents) ---------
def audit_machine(machine: str, mdir: Path, smoke: bool) -> tuple:
    rows = []
    for split in ("train", "test"):
        files = list_wavs(mdir, split)
        if smoke: files = files[:: max(1, len(files) // 20)]
        for f in files:
            r = parse_dcase_name(f.name); r.update(machine_type=machine, split_folder=split, path=str(f.relative_to(mdir)), size_bytes=f.stat().st_size,
                                                  readable=None, sr=None, channels=None, n_samples=None, duration_s=None, subtype=None, md5=None, error="")
            try:
                i = sf.info(str(f)); r.update(readable=True, sr=i.samplerate, channels=i.channels, n_samples=i.frames, duration_s=round(i.frames / i.samplerate, 3), subtype=i.subtype)
                if not smoke: r["md5"] = hashlib.md5(f.read_bytes()).hexdigest()
            except Exception as e:
                r.update(readable=False, error=str(e)[:150])
            rows.append(r)
    df = pd.DataFrame(rows)
    ok = df[df.readable == True]
    summary = dict(
        machine_type=machine, machine_dir=str(mdir), files_audited=len(df), unreadable=int((df.readable == False).sum()),
        train_total=int((df.split_folder == "train").sum()),
        train_source_normal=int(((df.split_folder == "train") & (df.domain == "source") & (df.label == "normal")).sum()),
        train_target_normal=int(((df.split_folder == "train") & (df.domain == "target") & (df.label == "normal")).sum()),
        train_not_normal=int(((df.split_folder == "train") & (df.label != "normal")).sum()),
        test_total=int((df.split_folder == "test").sum()),
        test_source=int(((df.split_folder == "test") & (df.domain == "source")).sum()), test_target=int(((df.split_folder == "test") & (df.domain == "target")).sum()),
        test_source_normal=int(((df.split_folder == "test") & (df.domain == "source") & (df.label == "normal")).sum()),
        test_source_anomaly=int(((df.split_folder == "test") & (df.domain == "source") & (df.label == "anomaly")).sum()),
        test_target_normal=int(((df.split_folder == "test") & (df.domain == "target") & (df.label == "normal")).sum()),
        test_target_anomaly=int(((df.split_folder == "test") & (df.domain == "target") & (df.label == "anomaly")).sum()),
        sample_rates=sorted(ok.sr.dropna().unique().tolist()), channels=sorted(ok.channels.dropna().unique().tolist()),
        duration_min_s=float(ok.duration_s.min()) if len(ok) else None, duration_max_s=float(ok.duration_s.max()) if len(ok) else None,
        n_distinct_durations=int(ok.duration_s.nunique()) if len(ok) else 0,
        duplicate_filenames=int(df.filename.duplicated().sum()), duplicate_contents=int(df.md5.dropna().duplicated().sum()) if not smoke else None,
        attribute_csv=[str(p.relative_to(mdir)) for p in mdir.rglob("*.csv")], attributes_in_names=sorted(df.attributes.dropna().unique().tolist())[:5],
        audited_all_files=not smoke,
    )
    problems = []
    if summary["unreadable"]: problems.append(f"{summary['unreadable']} unreadable files")
    if summary["train_not_normal"]: problems.append(f"{summary['train_not_normal']} non-normal files in train/ (excluded from training)")
    if not smoke:
        for k in ("train_source_normal", "train_target_normal", "test_source", "test_target"):
            if summary[k] != EXPECTED[k]: problems.append(f"{k}: expected {EXPECTED[k]}, observed {summary[k]}")
    if summary["sample_rates"] and summary["sample_rates"] != [COMMON_CONFIG["sample_rate"]]: problems.append(f"sample rates {summary['sample_rates']} (will be resampled)")
    if summary["channels"] and summary["channels"] != [2]: problems.append(f"channel counts {summary['channels']} (two-channel pipeline expects 2)")
    if summary["duplicate_filenames"]: problems.append(f"{summary['duplicate_filenames']} duplicate filenames")
    if summary["duplicate_contents"]: problems.append(f"{summary['duplicate_contents']} files with identical contents")
    summary["problems"] = problems
    return df, summary

In [ ]:
# ---- 3.3 Features, split, normalisation for one machine ------------------------------------------
def feature_signature(cfg):
    keys = ["sample_rate", "n_fft", "hop_length", "n_mels", "power", "channel_mode"]
    return hashlib.md5(json.dumps({k: cfg[k] for k in keys}, sort_keys=True).encode()).hexdigest()[:8]

def extract_and_cache(files, cfg, cache_dir: Path, name: str):
    sig = feature_signature(cfg); cache_dir.mkdir(parents=True, exist_ok=True)
    fp, op, lp = cache_dir / f"{name}_{sig}_frames.npy", cache_dir / f"{name}_{sig}_offsets.npy", cache_dir / f"{name}_{sig}_files.json"
    if fp.exists() and op.exists() and lp.exists() and json.loads(lp.read_text()) == [f.name for f in files]:
        log(f"  [{name}] feature cache hit"); return np.load(fp), np.load(op), files
    chunks, offsets, t0 = [], [0], time.time()
    for i, f in enumerate(files, 1):
        fr = wav_to_frames(f, cfg); chunks.append(fr); offsets.append(offsets[-1] + fr.shape[0])
        if i % 200 == 0 or i == len(files): log(f"  [{name}] extracted {i}/{len(files)} files, {offsets[-1]} frames, {time.time() - t0:.0f}s")
    frames = np.concatenate(chunks).astype(np.float32); offsets = np.asarray(offsets, np.int64)
    np.save(fp, frames); np.save(op, offsets); lp.write_text(json.dumps([f.name for f in files]))
    return frames, offsets, files

def make_split(src_files, tgt_files, cfg):
    rng = np.random.RandomState(cfg["seed"]); perm = rng.permutation(len(src_files))
    n_val = int(round(cfg["val_fraction"] * len(src_files))) if cfg["val_fraction"] > 0 else 0
    val_idx, tr_idx = np.sort(perm[:n_val]), np.sort(perm[n_val:])
    split = dict(seed=cfg["seed"], val_fraction=cfg["val_fraction"], n_source_available=len(src_files), n_target_available=len(tgt_files),
                 validation_source=[src_files[i].name for i in val_idx], modelA_train_source=[src_files[i].name for i in tr_idx],
                 modelB_train_source=[src_files[i].name for i in tr_idx], modelB_train_target=[f.name for f in tgt_files])
    return val_idx, tr_idx, split

class MachineData:
    # holds cached frames + split for one machine; builds datasets for A and B
    def __init__(self, machine, mdir, cfg, cache_dir, smoke):
        self.machine, self.cfg = machine, cfg
        train_all = list_wavs(mdir, "train")
        self.src_files = [f for f in train_all if parse_dcase_name(f.name)["domain"] == "source" and parse_dcase_name(f.name)["label"] == "normal"]
        self.tgt_files = [f for f in train_all if parse_dcase_name(f.name)["domain"] == "target" and parse_dcase_name(f.name)["label"] == "normal"]
        if smoke: self.src_files, self.tgt_files = self.src_files[:SMOKE_FILES], self.tgt_files[:min(3, len(self.tgt_files))]
        self.SRC, self.SRC_OFF, _ = extract_and_cache(self.src_files, cfg, cache_dir, f"{machine}_train_source")
        self.TGT, self.TGT_OFF, _ = extract_and_cache(self.tgt_files, cfg, cache_dir, f"{machine}_train_target") if self.tgt_files else (np.zeros((0, FRAME_DIM), np.float32), np.array([0]), [])
        self.val_idx, self.tr_idx, self.split = make_split(self.src_files, self.tgt_files, cfg)
    def build(self, use_target: bool):
        cfg = self.cfg
        parts = [self.SRC[self.SRC_OFF[i]:self.SRC_OFF[i + 1]] for i in self.tr_idx] + ([self.TGT] if use_target and len(self.tgt_files) else [])
        stats = fit_norm_stats(parts)
        src_n = torch.from_numpy(apply_norm(self.SRC, stats, cfg))
        frames_all, starts_all = src_n, window_starts(self.SRC_OFF, self.tr_idx, cfg["context_frames"], cfg["window_stride"])
        if use_target and len(self.tgt_files):
            tgt_n = torch.from_numpy(apply_norm(self.TGT, stats, cfg))
            frames_all = torch.cat([src_n, tgt_n]); starts_all = np.concatenate([starts_all, window_starts(self.TGT_OFF, np.arange(len(self.tgt_files)), cfg["context_frames"], cfg["window_stride"]) + src_n.shape[0]])
        val_ds = WindowDataset(src_n, window_starts(self.SRC_OFF, self.val_idx, cfg["context_frames"], 1), cfg["context_frames"])
        return WindowDataset(frames_all, starts_all, cfg["context_frames"]), val_ds, stats
    def windows_of(self, domain: str, i: int, stats: dict) -> np.ndarray:
        # normalised windows of one cached recording (no audio re-read)
        F, O = (self.SRC, self.SRC_OFF) if domain == "source" else (self.TGT, self.TGT_OFF)
        fr = apply_norm(F[O[i]:O[i + 1]], stats, self.cfg); n = fr.shape[0] - self.cfg["context_frames"] + 1
        idx = np.arange(n)[:, None] + np.arange(self.cfg["context_frames"])[None, :]
        return fr[idx].reshape(n, -1).astype(np.float32)

In [ ]:
# ---- 3.4 Training with resume, snapshots, completion markers (same loop as the ToyCar run) ---------
@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval(); tot, n = 0.0, 0
    for xb in loader:
        xb = xb.to(DEVICE); rec, _ = model(xb); tot += LOSS_FN(rec, xb).item() * xb.shape[0]; n += xb.shape[0]
    return tot / max(n, 1)
def _rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
def _set_rng_state(s):
    random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(s["cuda"])
def assert_mode_matches(ck, what):
    if ck.get("smoke_test") != SMOKE_TEST: raise RuntimeError(f"{what}: smoke_test={ck.get('smoke_test')} but this session is SMOKE_TEST={SMOKE_TEST}; never mix smoke and full outputs.")
VARIANT_DIR = {"source_only": "Source_Only", "source_target": "Source_Target"}

def train_variant(machine: str, variant: str, train_ds, val_ds, stats, cfg, train_files, validation_files):
    ckpt_dir = D["models"] / machine / VARIANT_DIR[variant]; ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path, done_path = ckpt_dir / f"{variant}_autoencoder.pth", ckpt_dir / f"{variant}_last.pth", ckpt_dir / f"{variant}_COMPLETE.json"
    hist_path = D["hist"] / f"{machine}_{variant}_training_history.csv"
    meta = dict(name=variant, smoke_test=SMOKE_TEST, config=cfg, input_dim=INPUT_DIM, frame_dim=FRAME_DIM, machine_type=machine, channel_labels=CHANNEL_LABELS,
                norm_mean=stats["mean"].tolist(), norm_std=stats["std"].tolist(), train_files=train_files, validation_files=validation_files,
                n_train_windows=len(train_ds), n_val_windows=len(val_ds), device=str(DEVICE), torch_version=torch.__version__)
    if done_path.exists() and best_path.exists():
        if not OVERWRITE_COMPLETED:
            ck = torch.load(best_path, map_location="cpu", weights_only=False); assert_mode_matches(ck, best_path.name)
            log(f"  [{machine}/{variant}] already COMPLETE (selected epoch {ck.get('best_epoch')}, {ck.get('checkpoint_selection', '')}); skipping."); return ck, pd.read_csv(hist_path)
        arch = ckpt_dir / f"archived_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"; arch.mkdir()
        for p in list(ckpt_dir.glob(f"{variant}_*")) + list(ckpt_dir.glob("epoch_*.pth")) + [hist_path]:
            if p.exists() and p.is_file(): shutil.move(str(p), str(arch / p.name))
        log(f"  [{machine}/{variant}] OVERWRITE_COMPLETED: previous files archived to {arch}")
    set_seed(cfg["seed"]); model = DCASEAutoencoder(INPUT_DIM).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    history, best_val, best_epoch, bad, start_epoch, elapsed_before = [], float("inf"), 0, 0, 1, 0.0
    if RESUME and last_path.exists():
        st = torch.load(last_path, map_location="cpu", weights_only=False); assert_mode_matches(st, last_path.name)
        core = lambda c: {k: v for k, v in c.items() if k not in ("epochs", "early_stop_patience", "snapshot_every")}
        if core(st["config"]) != core(cfg) or st["input_dim"] != INPUT_DIM: raise RuntimeError(f"{last_path.name} has different feature/model/data settings than COMMON_CONFIG.")
        model.load_state_dict(st["model_state_dict"]); opt.load_state_dict(st["optimizer_state_dict"])
        history, best_val, best_epoch, bad, start_epoch, elapsed_before = st["history"], st["best_val"], st["best_epoch"], st["bad_epochs"], st["epoch"] + 1, st.get("elapsed_seconds", 0.0)
        _set_rng_state(st["rng_state"]); model.to(DEVICE); log(f"  [{machine}/{variant}] RESUMING from epoch {st['epoch']}" + (f" (best val {best_val:.5f} at {best_epoch})" if len(val_ds) else " (spec_full: no validation set)"))
    else:
        meta["started"] = datetime.datetime.now().isoformat()
    tr_loader = make_loader(train_ds, cfg["batch_size"], True, True, cfg["seed"] + start_epoch)
    va_loader = make_loader(val_ds, INFER_BATCH, False, False, cfg["seed"]) if len(val_ds) else None
    log(f"  [{machine}/{variant}] {len(train_ds):,} train windows, {len(val_ds):,} val windows, epochs {start_epoch}..{cfg['epochs']}")
    t_sess = time.time()
    for epoch in range(start_epoch, cfg["epochs"] + 1):
        model.train(); t0, tot, n = time.time(), 0.0, 0
        for xb in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True); opt.zero_grad(set_to_none=True); rec, _ = model(xb); loss = LOSS_FN(rec, xb); loss.backward(); opt.step()
            tot += loss.item() * xb.shape[0]; n += xb.shape[0]
        tr_loss = tot / max(n, 1); va_loss = evaluate_loss(model, va_loader) if va_loader else float("nan")
        # spec_full: no validation set exists -> the checkpoint of the final epoch is the model (official-baseline behaviour)
        rec_ = dict(epoch=epoch, train_loss=tr_loss)
        if va_loader is not None: rec_["val_loss"] = va_loss          # spec_full: no validation set -> no validation column at all
        rec_.update(lr=opt.param_groups[0]["lr"], seconds=time.time() - t0); history.append(rec_)
        improved = (va_loader is None) or (va_loss < best_val - 1e-6)
        if improved:
            best_val, best_epoch, bad = (va_loss if va_loader else tr_loss), epoch, 0
            torch.save({**meta, "model_state_dict": model.state_dict(), "epoch": epoch, "best_epoch": epoch, "val_loss": va_loss, "train_loss": tr_loss}, best_path)
        else: bad += 1
        torch.save({**meta, "model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "epoch": epoch, "history": history, "best_val": best_val,
                    "best_epoch": best_epoch, "bad_epochs": bad, "rng_state": _rng_state(), "elapsed_seconds": elapsed_before + time.time() - t_sess}, last_path)
        if cfg["snapshot_every"] and epoch % cfg["snapshot_every"] == 0:
            torch.save({**meta, "model_state_dict": model.state_dict(), "epoch": epoch, "val_loss": va_loss, "train_loss": tr_loss}, ckpt_dir / f"epoch_{epoch:03d}.pth")
        pd.DataFrame(history).to_csv(hist_path, index=False)
        log(f"  [{machine}/{variant}] epoch {epoch:3d}/{cfg['epochs']} | train {tr_loss:.5f}" + (f" | val {va_loss:.5f}" if va_loader is not None else " | (no validation set)") + f" | {history[-1]['seconds']:.1f}s" + ("  *best*" if improved and va_loader is not None else ""))
        if cfg["early_stop_patience"] and va_loader and bad >= cfg["early_stop_patience"]:
            log(f"  [{machine}/{variant}] early stopping after {bad} epochs without improvement"); break
    total = elapsed_before + time.time() - t_sess
    ck = torch.load(best_path, map_location="cpu", weights_only=False)
    ck.update(finished=datetime.datetime.now().isoformat(), total_seconds=total, epochs_run=len(history), n_parameters=count_params(model), architecture=str(model),
              checkpoint_selection=("final epoch of fixed budget (no validation set)" if va_loader is None else "best validation loss (early stopping)"),
              has_validation_set=va_loader is not None); torch.save(ck, best_path)
    (ckpt_dir / f"{variant}_model_config.json").write_text(json.dumps({k: v for k, v in ck.items() if k not in ("model_state_dict",)}, indent=2, default=str))
    done_path.write_text(json.dumps(dict(completed=ck["finished"], epochs_run=len(history), best_epoch=best_epoch, best_val=best_val, smoke_test=SMOKE_TEST), indent=2))
    log(f"  [{machine}/{variant}] COMPLETE in {total/60:.1f} min | " + (f"final epoch {best_epoch} (spec_full: last epoch is the model) | final train loss {best_val:.5f}" if va_loader is None else f"best epoch {best_epoch} | best val {best_val:.5f}"))
    return ck, pd.DataFrame(history)

def plot_loss_curve(hist, title, out_png):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(hist.epoch, hist.train_loss, color="#2A6FDB", lw=2, label="training loss")
    if "val_loss" in hist and hist.val_loss.notna().any(): ax.plot(hist.epoch, hist.val_loss, color="#E07B2A", lw=2, label="validation loss")
    else: ax.text(0.99, 0.95, "no held-out validation set (spec_full)", transform=ax.transAxes, ha="right", va="top", fontsize=8, color="#666666")
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE on this model's normalised features"); ax.set_title(title, loc="left"); ax.grid(alpha=0.25); ax.legend(frameon=False)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.tight_layout(); fig.savefig(out_png, dpi=140); plt.close(fig)

def plot_two_panel(hA, hB, machine, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
    for ax, h, ttl in zip(axes, [hA, hB], ["Model A: source only", "Model B: source + target"]):
        ax.plot(h.epoch, h.train_loss, color="#2A6FDB", lw=2, label="training")
        if "val_loss" in h and h.val_loss.notna().any(): ax.plot(h.epoch, h.val_loss, color="#E07B2A", lw=2, label="validation")
        else: ax.text(0.99, 0.95, "no held-out validation set (spec_full)", transform=ax.transAxes, ha="right", va="top", fontsize=8, color="#666666")
        ax.set_title(f"{machine} - {ttl} (own normalisation scale)", loc="left"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE, normalised features"); ax.grid(alpha=0.25); ax.legend(frameon=False)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.suptitle("Separate y-scales on purpose: A and B use different normalisation statistics, so their losses are not directly comparable", fontsize=9, y=1.02)
    fig.tight_layout(); fig.savefig(out_png, dpi=140, bbox_inches="tight"); plt.close(fig)

In [ ]:
# ---- 3.5 Experiment 2 preparation: Mahalanobis statistics of latent (8-d) and residual frames (256-d) ----
class RunningStats:
    # exact mean and covariance via float64 accumulators (no need to hold all vectors in memory)
    def __init__(self, d): self.n, self.s1, self.s2 = 0, torch.zeros(d, dtype=torch.float64, device=DEVICE), torch.zeros(d, d, dtype=torch.float64, device=DEVICE)
    def add(self, x):  # x: (n, d) tensor on DEVICE
        x = x.double(); self.n += x.shape[0]; self.s1 += x.sum(0); self.s2 += x.T @ x
    def finish(self):
        mu = self.s1 / self.n; cov = (self.s2 - self.n * torch.outer(mu, mu)) / max(self.n - 1, 1)
        cov = 0.5 * (cov + cov.T)
        return mu.cpu().numpy(), cov.cpu().numpy(), self.n

def regularised_precision(cov: np.ndarray, rel_eps: float = 1e-3) -> tuple:
    # Sigma_reg = Sigma + lambda*I with lambda = rel_eps * trace(Sigma)/d. lambda scales with the data, so it is never negligible
    # relative to the largest variance and never dominant. Cholesky factorisation proves positive definiteness; the inverse is obtained
    # by solving L L^T X = I (scipy cho_solve), which is more stable than a general inverse. Precision is symmetrised afterwards.
    import scipy.linalg
    d = cov.shape[0]; lam = rel_eps * np.trace(cov) / d
    if not np.isfinite(lam) or lam <= 0: lam = 1e-6
    cov_reg = cov + lam * np.eye(d)
    L = np.linalg.cholesky(cov_reg)                    # raises LinAlgError if not positive definite
    prec = scipy.linalg.cho_solve((L, True), np.eye(d)); prec = 0.5 * (prec + prec.T)
    return cov_reg, prec, float(lam), L

@torch.no_grad()
def mahalanobis_statistics(machine: str, variant: str, ck: dict, data: MachineData) -> dict:
    out_dir = D["mahala"] / machine; out_dir.mkdir(parents=True, exist_ok=True)
    done = out_dir / f"{variant}_mahalanobis_COMPLETE.json"
    if done.exists() and not OVERWRITE_COMPLETED:
        log(f"  [{machine}/{variant}] Mahalanobis statistics already present; skipping."); return json.loads(done.read_text())
    model = DCASEAutoencoder(ck["input_dim"]); model.load_state_dict(ck["model_state_dict"]); model.eval().to(DEVICE)
    stats = dict(mean=np.asarray(ck["norm_mean"], np.float32), std=np.asarray(ck["norm_std"], np.float32))
    name_to_idx_src = {f.name: i for i, f in enumerate(data.src_files)}; name_to_idx_tgt = {f.name: i for i, f in enumerate(data.tgt_files)}
    # Groups of GENUINE NORMAL TRAINING recordings (never test files):
    #   source_train      : the source normals this model trained on (A and B: identical list)
    #   target_train      : the 10 target normals from train/ (B trained on them; A did NOT - they are still normal training-split clips, as used by the
    #                       official baseline for its cov_target)
    #   all_train         : source_train + target_train (= B's training set)
    #   validation_source : only under holdout_10pct - the 99 held-out source normals (never trained on); absent under spec_full
    groups = {"source_train": [("source", name_to_idx_src[n]) for n in ck["train_files"]["source"]],
              "target_train": [("target", name_to_idx_tgt[n]) for n in data.split["modelB_train_target"]]}
    groups["all_train"] = groups["source_train"] + groups["target_train"]
    if ck["validation_files"]: groups["validation_source"] = [("source", name_to_idx_src[n]) for n in ck["validation_files"]]
    for gname, items in groups.items():
        names = [(data.src_files if d == "source" else data.tgt_files)[i].name for d, i in items]
        assert all(parse_dcase_name(n)["split"] == "train" and parse_dcase_name(n)["label"] == "normal" for n in names), f"{gname}: non-training or non-normal file in group"
        assert not any("_test_" in n for n in names), f"{gname}: test file in group"
    trained_on = {"source_only": ["source_train"], "source_target": ["source_train", "target_train", "all_train"]}[variant]
    report = dict(machine=machine, variant=variant, checkpoint=f"Models/{machine}/{VARIANT_DIR[variant]}/{variant}_autoencoder.pth", latent_dim=8, residual_dim=ck["frame_dim"],
                  regularisation="Sigma_reg = Sigma + lambda*I, lambda = 1e-3 * trace(Sigma)/d; precision = inverse via Cholesky solve",
                  group_definitions={g: dict(n_recordings=len(items), model_trained_on_these=(g in trained_on), split="train", label="normal") for g, items in groups.items()},
                  groups={}, generated=datetime.datetime.now().isoformat(), smoke_test=SMOKE_TEST)
    t0 = time.time()
    for gname, items in groups.items():
        if not items: continue
        lat, res = RunningStats(8), RunningStats(ck["frame_dim"]); n_windows = 0
        for dom, i in items:
            x = torch.from_numpy(data.windows_of(dom, i, stats)).to(DEVICE)
            for s in range(0, len(x), INFER_BATCH):
                xb = x[s:s + INFER_BATCH]; rec, z = model(xb)
                lat.add(z); res.add((xb - rec).reshape(-1, ck["frame_dim"])); n_windows += xb.shape[0]
        lmu, lcov, ln = lat.finish(); rmu, rcov, rn = res.finish()
        lcov_r, lprec, llam, _ = regularised_precision(lcov); rcov_r, rprec, rlam, _ = regularised_precision(rcov)
        fd = ck["frame_dim"]
        checks = dict(latent_mean_shape=lmu.shape == (8,), latent_cov_shape=lcov.shape == (8, 8), latent_precision_shape=lprec.shape == (8, 8),
                      residual_mean_shape=rmu.shape == (fd,), residual_cov_shape=rcov.shape == (fd, fd), residual_precision_shape=rprec.shape == (fd, fd),
                      latent_finite=bool(np.isfinite(lprec).all() and np.isfinite(lmu).all() and np.isfinite(lcov).all()), residual_finite=bool(np.isfinite(rprec).all() and np.isfinite(rmu).all() and np.isfinite(rcov).all()),
                      latent_min_eig_reg=float(np.linalg.eigvalsh(lcov_r).min()), residual_min_eig_reg=float(np.linalg.eigvalsh(rcov_r).min()),
                      latent_cond_reg=float(np.linalg.cond(lcov_r)), residual_cond_reg=float(np.linalg.cond(rcov_r)),
                      latent_inverse_error=float(np.abs(lprec @ lcov_r - np.eye(8)).max()), residual_inverse_error=float(np.abs(rprec @ rcov_r - np.eye(fd)).max()),
                      latent_symmetric=bool(np.allclose(lprec, lprec.T, atol=1e-6)), residual_symmetric=bool(np.allclose(rprec, rprec.T, atol=1e-4)),
                      n_samples_latent=int(ln), n_samples_residual=int(rn), enough_samples=bool(ln > 8 * 10 and rn > fd * 10))
        checks["ok"] = bool(all(checks[k] for k in checks if k.endswith("_shape")) and checks["latent_finite"] and checks["residual_finite"] and checks["latent_min_eig_reg"] > 0 and checks["residual_min_eig_reg"] > 0
                            and checks["latent_inverse_error"] < 1e-6 and checks["residual_inverse_error"] < 1e-3 and checks["latent_symmetric"] and checks["residual_symmetric"] and checks["enough_samples"])
        np.savez(out_dir / f"{variant}_{gname}_mahalanobis_stats.npz",
                 latent_mean=lmu.astype(np.float32), latent_cov=lcov.astype(np.float32), latent_cov_regularised=lcov_r.astype(np.float32), latent_precision=lprec.astype(np.float32),
                 residual_mean=rmu.astype(np.float32), residual_cov=rcov.astype(np.float32), residual_cov_regularised=rcov_r.astype(np.float32), residual_precision=rprec.astype(np.float32),
                 latent_lambda=llam, residual_lambda=rlam, n_windows=n_windows, n_residual_frames=rn, n_recordings=len(items))
        report["groups"][gname] = dict(n_recordings=len(items), n_windows=int(n_windows), n_residual_frames=int(rn), latent_lambda=llam, residual_lambda=rlam, checks=checks,
                                       file=f"Mahalanobis_Statistics/{machine}/{variant}_{gname}_mahalanobis_stats.npz")
    report["all_checks_ok"] = all(g["checks"]["ok"] for g in report["groups"].values()); report["seconds"] = time.time() - t0
    (out_dir / f"{variant}_mahalanobis_report.json").write_text(json.dumps(report, indent=2))
    if report["all_checks_ok"]: done.write_text(json.dumps(report, indent=2))
    log(f"  [{machine}/{variant}] Mahalanobis statistics for {list(report['groups'])} in {report['seconds']:.0f}s | numerically valid: {report['all_checks_ok']}")
    return report

In [ ]:
# ---- 3.5b Optional cross-fitted calibration: out-of-fold scores of SOURCE normals (independent of any model that saw them) ----
def calibration_scores(machine: str, data: MachineData):
    # NOTE: not decorated with no_grad because it trains fold models; inference parts use torch.no_grad() explicitly
    out_csv = D["calib"] / f"{machine}_out_of_fold_normal_scores.csv"; done = D["calib"] / f"{machine}_calibration_COMPLETE.json"
    if done.exists() and not OVERWRITE_COMPLETED: log(f"  [{machine}] calibration scores already present; skipping."); return
    K = CALIBRATION_FOLDS; cfg = {**COMMON_CONFIG, "epochs": CALIBRATION_EPOCHS}
    rng = np.random.RandomState(cfg["seed"]); folds = np.array_split(rng.permutation(len(data.src_files)), K)
    rows = []
    for k, held in enumerate(folds, 1):
        held = np.sort(held); tr_idx = np.sort(np.setdiff1d(np.arange(len(data.src_files)), held))
        variant = f"calib_fold{k}"; VARIANT_DIR[variant] = f"Calibration/fold_{k}"
        parts = [data.SRC[data.SRC_OFF[i]:data.SRC_OFF[i + 1]] for i in tr_idx]; stats = fit_norm_stats(parts)
        src_n = torch.from_numpy(apply_norm(data.SRC, stats, cfg))
        tr_ds = WindowDataset(src_n, window_starts(data.SRC_OFF, tr_idx, cfg["context_frames"], cfg["window_stride"]), cfg["context_frames"])
        empty = WindowDataset(src_n, np.zeros(0, np.int64), cfg["context_frames"])
        ck, _ = train_variant(machine, variant, tr_ds, empty, stats, cfg, dict(source=[data.src_files[i].name for i in tr_idx], target=[]), [])
        # statistics of this fold model on ITS training normals, then score the held-out fold
        model = DCASEAutoencoder(ck["input_dim"]); model.load_state_dict(ck["model_state_dict"]); model.eval().to(DEVICE)
        torch.set_grad_enabled(False)
        lat, res = RunningStats(8), RunningStats(FRAME_DIM)
        for i in tr_idx:
            x = torch.from_numpy(data.windows_of("source", i, stats)).to(DEVICE)
            for s0 in range(0, len(x), INFER_BATCH):
                xb = x[s0:s0 + INFER_BATCH]; rec, z = model(xb); lat.add(z); res.add((xb - rec).reshape(-1, FRAME_DIM))
        lmu, lcov, _ = lat.finish(); rmu, rcov, _ = res.finish(); _, lP, _, _ = regularised_precision(lcov); _, rP, _, _ = regularised_precision(rcov)
        lP_t, rP_t = torch.from_numpy(lP).float().to(DEVICE), torch.from_numpy(rP).float().to(DEVICE); lmu_t, rmu_t = torch.from_numpy(lmu).float().to(DEVICE), torch.from_numpy(rmu).float().to(DEVICE)
        for i in held:
            x = torch.from_numpy(data.windows_of("source", i, stats)).to(DEVICE); rec, z = model(x); r = (x - rec)
            dl = z - lmu_t; dr = r.reshape(-1, FRAME_DIM) - rmu_t
            rows.append(dict(machine=machine, filename=data.src_files[i].name, fold=k, mse_mean=float((r ** 2).mean()), mse_max_window=float((r ** 2).mean(1).max()),
                             mahala_latent_mean=float(torch.einsum("ni,ij,nj->n", dl, lP_t, dl).mean()), mahala_residual_mean=float(torch.einsum("ni,ij,nj->n", dr, rP_t, dr).mean())))
        torch.set_grad_enabled(True)
        log(f"  [{machine}] calibration fold {k}/{K}: {len(held)} held-out source normals scored")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    done.write_text(json.dumps(dict(machine=machine, folds=K, epochs=CALIBRATION_EPOCHS, n_scored=len(rows), note="out-of-fold scores of source training normals from fold models that never saw them; "
                                   "fold models are NOT the deliverable models; use the score distributions as independent normal references for threshold selection", generated=datetime.datetime.now().isoformat()), indent=2))
    log(f"  [{machine}] calibration: {len(rows)} independent normal scores -> {out_csv.name}")

# ---- 3.6 Verification of one model (checkpoint, dims, NaN/inf, real audio, norm stats, Mahalanobis stats, reproducibility) ----
def verify_model(machine: str, variant: str, mdir: Path, data: MachineData) -> dict:
    ck_path = D["models"] / machine / VARIANT_DIR[variant] / f"{variant}_autoencoder.pth"
    r = dict(machine=machine, variant=variant, checkpoint_exists=ck_path.exists())
    try:
        model, ck = load_checkpoint(ck_path); assert_mode_matches(ck, ck_path.name)
        r.update(loads=True, architecture_ok=(count_params(model) == 432408 and model.latent_dim == 8), input_dim=ck["input_dim"], input_dim_ok=(ck["input_dim"] == INPUT_DIM))
        x = torch.randn(8, ck["input_dim"]); rec, z = model(x)
        r.update(output_dim_ok=(rec.shape == x.shape), latent_dim_ok=(z.shape == (8, 8)), no_nan_random=bool(not torch.isnan(rec).any()), no_inf_random=bool(torch.isfinite(rec).all()))
        files = [data.src_files[i] for i in data.tr_idx[:2]] + [data.src_files[i] for i in data.val_idx[:1]] + data.tgt_files[:1]
        finite, repro, mses = True, True, []
        for f in files:
            xi = wav_to_model_input(f, ck); a = run_inference(model, xi); b = run_inference(model, xi)
            finite &= bool(np.isfinite(a["reconstruction"]).all() and np.isfinite(a["latent"]).all()); repro &= bool(np.array_equal(a["reconstruction"], b["reconstruction"])); mses.append(float(a["window_mse"].mean()))
        r.update(real_recordings_tested=len(files), finite_on_real_audio=finite, reproducible=repro, mean_window_mse_normal=float(np.mean(mses)))
        npz = np.load(D["pre_cfg"] / f"{machine}_{variant}_feature_norm_stats.npz")
        r["norm_stats_loadable"] = bool(np.allclose(npz["mean"], ck["norm_mean"]) and np.allclose(npz["std"], ck["norm_std"]))
        rep = json.loads((D["mahala"] / machine / f"{variant}_mahalanobis_report.json").read_text())
        r["mahalanobis_stats_valid"] = bool(rep["all_checks_ok"])
        s = np.load(D["mahala"] / machine / f"{variant}_source_train_mahalanobis_stats.npz")
        r["mahalanobis_dims_ok"] = (s["latent_precision"].shape == (8, 8) and s["residual_precision"].shape == (ck["frame_dim"], ck["frame_dim"]))
        r["passed"] = all(bool(r[k]) for k in ("loads", "architecture_ok", "input_dim_ok", "output_dim_ok", "latent_dim_ok", "no_nan_random", "no_inf_random", "finite_on_real_audio", "reproducible", "norm_stats_loadable", "mahalanobis_stats_valid", "mahalanobis_dims_ok"))
    except Exception as e:
        r.update(passed=False, error=f"{type(e).__name__}: {e}")
    return r

## Section 4 — Stage 1: verify and inspect all seven archives (no extraction)
Expected output: one row per archive with size, the machine folder name inside the ZIP, train/test counts by domain and label, supplemental and attribute-CSV presence. Missing archives are listed with the exact Drive path where they are expected; the notebook continues with the archives that are present and will process the others on a later run.

In [ ]:
# ---- 4.1 Archive existence and internal structure ---------------------------------------------------
ARCHIVE_INFO = {}
for m, zname in ARCHIVES.items():
    zp = DRIVE_DATA_DIR / zname
    info = inspect_archive(zp); info["canonical_machine"] = m; info["path"] = str(zp); ARCHIVE_INFO[m] = info
    if not info["exists"]:
        mark(STATE, m, "archive_present", False, note=f"archive missing at {zp}", status="MISSING")
    else:
        if STATE[m]["status"] in ("MISSING", "pending"): STATE[m]["status"] = "pending"
        mark(STATE, m, "archive_present", True)
cols = ["canonical_machine", "zip", "exists", "size_mb", "machine_dir_in_zip", "train_source_normal", "train_target_normal", "train_other", "test_source_normal", "test_source_anomaly",
        "test_target_normal", "test_target_anomaly", "supplemental_wavs", "other_wavs", "duplicate_filenames", "attribute_csvs"]
ARCH_DF = pd.DataFrame([{c: ARCHIVE_INFO[m].get(c) for c in cols} for m in MACHINE_ORDER]).set_index("canonical_machine")
display(ARCH_DF)
(D["audit"] / "archive_structure.json").write_text(json.dumps(ARCHIVE_INFO, indent=2, default=str))
missing = [m for m in MACHINE_ORDER if not ARCHIVE_INFO[m]["exists"]]
present = [m for m in MACHINE_ORDER if ARCHIVE_INFO[m]["exists"]]
log(f"archives present: {present} | missing: {missing}")
if missing:
    print("\nMISSING ARCHIVES - upload these to Google Drive with exactly these names, then re-run:")
    for m in missing: print(f"   {DRIVE_DATA_DIR / ARCHIVES[m]}")
if not present: raise SystemExit("STOP: no archive found. Upload the ZIP files to Drive first (see the instructions at the top).")
for m in present:
    i = ARCHIVE_INFO[m]
    dev = [k for k in ("train_source_normal", "train_target_normal") if i[k] != EXPECTED[k]] + [k for k in ("test_source", "test_target") if i[k] != EXPECTED[k]]
    if dev: print(f"NOTE {m}: counts differ from the project document for {dev} -> the full audit after extraction decides (STRICT_COUNTS={STRICT_COUNTS}).")

## Section 5 — Stage 2: check existing ToyCar models and preserve the earlier run
The earlier ToyCar run (`DCASE_2026_PART1_RESULTS/`) trained Model A on 891 source normals and Model B on 891 source + 10 target normals, with 99 source normals held out for validation (holdout protocol). This stage compares those checkpoints with the protocol selected in CONFIG and never writes to that folder.

1. **Under `spec_full` (approved):** the earlier models are **incompatible** because they used a different split protocol (891 / 901 recordings with a validation hold-out instead of 990 / 1000 with none). They are copied unchanged to `Models/ToyCar/Previous_Holdout_Validation_Run/` as **reference models only** and are **not** used as the primary deliverables. New ToyCar models are trained separately in Stage 3 under `spec_full`: Model A on all 990 source normals, Model B on all 990 source + 10 target normals, fixed 100 epochs, final-epoch checkpoint.
2. **Under `holdout_10pct`:** existing ToyCar models whose feature settings, architecture, input dimension and split protocol match are reused as they are (copied into `Models/ToyCar/`, not retrained).

In both cases the cell prints the exact comparison result and reasons.

In [ ]:
# ---- 5.1 ToyCar compatibility check and copy -----------------------------------------------------------
def integrate_existing_toycar() -> dict:
    res = dict(found=False, compatible=False, reasons=[])
    src_root = EXISTING_TOYCAR_RESULTS
    core = lambda c: {k: v for k, v in c.items() if k not in ("epochs", "early_stop_patience", "snapshot_every")}
    for variant, sub in (("source_only", "source_only_experiment"), ("source_target", "source_target_experiment")):
        ck_p = src_root / sub / "checkpoints" / f"{variant}_autoencoder.pth"; done_p = src_root / sub / "checkpoints" / f"{variant}_COMPLETE.json"
        hist_p = src_root / sub / "training_history" / f"{variant}_training_history.csv"; curve_p = src_root / sub / "graphs" / f"{variant}_loss_curve.png"
        cfg_p = src_root / sub / "checkpoints" / f"{variant}_model_config.json"
        if not ck_p.exists(): res["reasons"].append(f"{ck_p} not found"); continue
        res["found"] = True
        ck = torch.load(ck_p, map_location="cpu", weights_only=False)
        probs = []
        if ck.get("smoke_test"): probs.append("checkpoint is a smoke-test checkpoint")
        if not done_p.exists(): probs.append("no COMPLETE marker (training unfinished)")
        feat_keys = ("sample_rate", "n_fft", "hop_length", "n_mels", "context_frames", "power", "channel_mode", "normalise", "window_stride", "batch_size", "learning_rate", "seed")
        if any(ck["config"].get(k) != COMMON_CONFIG[k] for k in feat_keys): probs.append(f"feature/model config differs: {[k for k in feat_keys if ck['config'].get(k) != COMMON_CONFIG[k]]}")
        if float(ck["config"].get("val_fraction", 0.0)) != float(COMMON_CONFIG["val_fraction"]):
            probs.append(f"split protocol differs: existing run held out {len(ck['validation_files'])} source recordings (trained on {len(ck['train_files']['source'])}+{len(ck['train_files']['target'])}), "
                         f"SPLIT_PROTOCOL={SPLIT_PROTOCOL!r} requires val_fraction={COMMON_CONFIG['val_fraction']}")
        if ck["input_dim"] != INPUT_DIM or ck["frame_dim"] != FRAME_DIM: probs.append(f"input_dim {ck['input_dim']} / frame_dim {ck['frame_dim']} differ from {INPUT_DIM}/{FRAME_DIM}")
        if not hist_p.exists(): probs.append("training history missing")
        if probs: res["reasons"].append(f"{variant}: " + "; ".join(probs)); continue
        dest = D["models"] / "ToyCar" / VARIANT_DIR[variant]; dest.mkdir(parents=True, exist_ok=True)
        if not (dest / f"{variant}_COMPLETE.json").exists():
            shutil.copy2(ck_p, dest / f"{variant}_autoencoder.pth"); shutil.copy2(done_p, dest / f"{variant}_COMPLETE.json")
            if cfg_p.exists(): shutil.copy2(cfg_p, dest / f"{variant}_model_config.json")
            shutil.copy2(hist_p, D["hist"] / f"ToyCar_{variant}_training_history.csv")
            if curve_p.exists(): shutil.copy2(curve_p, D["curves"] / f"ToyCar_{variant}_loss_curve.png")
            (dest / "REUSED_FROM.txt").write_text(f"Copied (not moved) from {ck_p} on {datetime.datetime.now().isoformat()}. Original left untouched.\n")
        res[variant] = dict(best_epoch=ck.get("best_epoch"), epochs_run=ck.get("epochs_run"), train=len(ck["train_files"]["source"]) + len(ck["train_files"]["target"]), finished=ck.get("finished"))
    res["compatible"] = res["found"] and not res["reasons"] and all(v in res for v in ("source_only", "source_target"))
    if res["found"] and not res["compatible"]:
        # keep the genuine previous run as a read-only reference inside the project (never modified, never used as the primary deliverable)
        ref = D["models"] / "ToyCar" / "Previous_Holdout_Validation_Run"
        if not ref.exists():
            ref.mkdir(parents=True)
            for variant, sub in (("source_only", "source_only_experiment"), ("source_target", "source_target_experiment")):
                for fp in [src_root / sub / "checkpoints" / f"{variant}_autoencoder.pth", src_root / sub / "checkpoints" / f"{variant}_model_config.json", src_root / sub / "training_history" / f"{variant}_training_history.csv", src_root / sub / "graphs" / f"{variant}_loss_curve.png"]:
                    if fp.exists(): shutil.copy2(fp, ref / fp.name)
            (ref / "README.txt").write_text("Reference copy of the earlier ToyCar run (891 source / 891+10 training, 99 validation, early stopping). Copied unchanged from " + str(src_root) + ". Not the spec_full deliverable.\n")
    return res

if SMOKE_TEST:
    TOYCAR_REUSE = dict(found=False, compatible=False, reasons=["smoke mode never reuses full-training checkpoints"])
else:
    TOYCAR_REUSE = integrate_existing_toycar()
print(json.dumps(TOYCAR_REUSE, indent=2, default=str))
if TOYCAR_REUSE["compatible"]:
    mark(STATE, "ToyCar", "models_reused_from_existing_run", True, note="A and B copied from DCASE_2026_PART1_RESULTS; not retrained")
    log("ToyCar: existing full-training checkpoints are compatible and were copied into Models/ToyCar (originals untouched).")
else:
    log("ToyCar: existing checkpoints NOT reused as the primary deliverable (" + "; ".join(TOYCAR_REUSE["reasons"]) + "). ToyCar will be trained under SPLIT_PROTOCOL="
        + repr(SPLIT_PROTOCOL) + " into Models/ToyCar; the original results folder is not modified" + (" and a reference copy sits in Models/ToyCar/Previous_Holdout_Validation_Run." if TOYCAR_REUSE["found"] else "."))

## Section 6 — Stage 3: process every machine, one at a time
Per machine: extract (train/test only) → audit all files → cache features → split → Model A → Model B → Mahalanobis statistics for both → verification → per-machine report → delete local files → mark `COMPLETE`. A machine whose audit fails the expected counts (with `STRICT_COUNTS = True`) is marked `INCOMPLETE` with the reason and skipped; nothing is fabricated. Expected output: a long log, one line per epoch. On a T4 this cell runs for roughly 35 to 50 minutes per machine; if Colab disconnects, re-run the notebook and it continues.

In [ ]:
# ---- 6.1 The sequential loop ---------------------------------------------------------------------------
def required_files_for(machine: str) -> list:
    req = [D["audit"] / f"dataset_audit_{machine}.csv", D["audit"] / f"dataset_summary_{machine}.json", D["pre_cfg"] / f"{machine}_split_manifest.json",
           D["pre_cfg"] / f"{machine}_preprocessing_config.json", D["pre_cfg"] / f"{machine}_feature_verification.json", D["curves"] / f"{machine}_training_curves_per_model.png",
           D["verify"] / f"{machine}_verification.csv", D["verify"] / f"{machine}_verification.md"]
    for v in ("source_only", "source_target"):
        req += [D["models"] / machine / VARIANT_DIR[v] / f"{v}_autoencoder.pth", D["models"] / machine / VARIANT_DIR[v] / f"{v}_model_config.json", D["models"] / machine / VARIANT_DIR[v] / f"{v}_COMPLETE.json", D["models"] / machine / VARIANT_DIR[v] / "TRAINING_PROTOCOL.txt",
                D["pre_cfg"] / f"{machine}_{v}_feature_norm_stats.npz", D["hist"] / f"{machine}_{v}_training_history.csv", D["curves"] / f"{machine}_{v}_loss_curve.png",
                D["mahala"] / machine / f"{v}_mahalanobis_report.json", D["mahala"] / machine / f"{v}_mahalanobis_COMPLETE.json"]
        req += [D["mahala"] / machine / f"{v}_{g}_mahalanobis_stats.npz" for g in ("source_train", "target_train", "all_train")]
    return req

def process_machine(machine: str):
    st = STATE[machine]
    if st["status"] == "COMPLETE" and not OVERWRITE_COMPLETED:
        log(f"=== {machine}: already COMPLETE, skipping ==="); return
    if not ARCHIVE_INFO[machine]["exists"]:
        log(f"=== {machine}: archive missing, skipping ==="); return
    log(f"=== {machine}: START ({mem_info()}; {disk_info(LOCAL_WORK)}) ===")
    zp = DRIVE_DATA_DIR / ARCHIVES[machine]; local = LOCAL_WORK / "extract" / machine
    cache_dir = (D["pre_cfg"].parent / "feature_cache" / machine) if FEATURE_CACHE_ON_DRIVE else (LOCAL_WORK / "cache" / machine)
    # 1. extract
    extract_machine(zp, local); mdir = find_machine_dir(local); mark(STATE, machine, "extracted", True, status="in_progress")
    # 2. audit
    audit_df, summary = audit_machine(machine, mdir, SMOKE_TEST)
    audit_df.to_csv(D["audit"] / f"dataset_audit_{machine}.csv", index=False)
    (D["audit"] / f"dataset_summary_{machine}.json").write_text(json.dumps(summary, indent=2, default=str))
    log(f"  audit: {summary['files_audited']} files | src/tgt train {summary['train_source_normal']}/{summary['train_target_normal']} | test s/t {summary['test_source']}/{summary['test_target']} "
        f"(normal/anomaly src {summary['test_source_normal']}/{summary['test_source_anomaly']}, tgt {summary['test_target_normal']}/{summary['test_target_anomaly']}) | sr {summary['sample_rates']} | ch {summary['channels']} | "
        f"dur {summary['duration_min_s']}..{summary['duration_max_s']}s | problems: {summary['problems'] or 'none'}")
    blocking = [p for p in summary["problems"] if p.startswith(("train_source_normal", "train_target_normal", "test_source", "test_target")) or "unreadable" in p or "channel counts" in p]
    if blocking and STRICT_COUNTS and not SMOKE_TEST:
        mark(STATE, machine, "audit", False, note="INCOMPLETE: " + "; ".join(blocking), status="INCOMPLETE")
        log(f"  {machine} marked INCOMPLETE (STRICT_COUNTS): {blocking}. Not trained. Fix the archive or set STRICT_COUNTS=False to train on what is present."); return
    mark(STATE, machine, "audit", True, note=("audit warnings: " + "; ".join(summary["problems"])) if summary["problems"] else None)
    # 3. features + split
    data = MachineData(machine, mdir, COMMON_CONFIG, cache_dir, SMOKE_TEST)
    split_doc = {**data.split, "machine": machine, "actual_counts": dict(validation=len(data.split["validation_source"]), modelA_train=len(data.split["modelA_train_source"]),
                 modelB_train=len(data.split["modelB_train_source"]) + len(data.split["modelB_train_target"]))}
    (D["pre_cfg"] / f"{machine}_split_manifest.json").write_text(json.dumps(split_doc, indent=2))
    fv = dict(machine=machine, frame_dim=FRAME_DIM, input_dim=INPUT_DIM, source_frames=int(data.SRC.shape[0]), target_frames=int(data.TGT.shape[0]), frames_per_recording_min=int(np.diff(data.SRC_OFF).min()),
              frames_per_recording_max=int(np.diff(data.SRC_OFF).max()), all_finite=bool(np.isfinite(data.SRC).all() and np.isfinite(data.TGT).all()), feature_signature=feature_signature(COMMON_CONFIG),
              windows_per_recording=int(np.diff(data.SRC_OFF).min()) - COMMON_CONFIG["context_frames"] + 1)
    (D["pre_cfg"] / f"{machine}_feature_verification.json").write_text(json.dumps(fv, indent=2)); (D["pre_cfg"] / f"{machine}_preprocessing_config.json").write_text(json.dumps(COMMON_CONFIG, indent=2))
    log(f"  split [{SPLIT_PROTOCOL}]: Model A gradient updates on {len(data.tr_idx)} source | Model B on {len(data.tr_idx)} source + {len(data.tgt_files)} target | validation {len(data.val_idx)} source"
        + (" (none: all recordings train, final epoch = model)" if len(data.val_idx) == 0 else " (held out, shared by A and B)") + f" | frames finite: {fv['all_finite']}")
    mark(STATE, machine, "features", True)
    # 4 + 5. train A and B (skipped automatically when COMPLETE markers exist, e.g. reused ToyCar)
    cks, hists = {}, {}
    for variant, use_target in (("source_only", False), ("source_target", True)):
        tr_ds, va_ds, stats = data.build(use_target)
        tf = dict(source=data.split["modelA_train_source"], target=data.split["modelB_train_target"] if use_target else [])
        ck, hist = train_variant(machine, variant, tr_ds, va_ds, stats, COMMON_CONFIG, tf, data.split["validation_source"])
        if ck["train_files"] != tf or ck["validation_files"] != data.split["validation_source"]:
            raise RuntimeError(f"{machine}/{variant}: the checkpoint's recorded split differs from the split recomputed now (SPLIT_PROTOCOL or file list changed?). "
                               "Use the same SPLIT_PROTOCOL as the existing checkpoints, or set OVERWRITE_COMPLETED=True to retrain.")
        np.savez(D["pre_cfg"] / f"{machine}_{variant}_feature_norm_stats.npz", mean=np.asarray(ck["norm_mean"], np.float32), std=np.asarray(ck["norm_std"], np.float32))
        shutil.copy2(D["models"] / machine / VARIANT_DIR[variant] / f"{variant}_model_config.json", D["cfg"] / f"{machine}_{variant}_model_config.json")
        if not (D["curves"] / f"{machine}_{variant}_loss_curve.png").exists(): plot_loss_curve(hist, f"{machine} - {variant}", D["curves"] / f"{machine}_{variant}_loss_curve.png")
        cks[variant], hists[variant] = ck, hist; mark(STATE, machine, f"train_{variant}", True)
        del tr_ds, va_ds
    plot_two_panel(hists["source_only"], hists["source_target"], machine, D["curves"] / f"{machine}_training_curves_per_model.png")
    note = (f"TRAINING PROTOCOL: {SPLIT_PROTOCOL}\n" + ("No independent held-out validation set exists. Model A used all {a} source normals and Model B all {a} source + {t} target normals for gradient "
            "updates; fixed budget of {e} epochs; the checkpoint is the FINAL epoch. No early stopping and no validation loss were used or recorded. Training loss is the only "
            "monitored quantity and is NOT an estimate of generalisation.\n".format(a=len(data.tr_idx), t=len(data.tgt_files), e=COMMON_CONFIG["epochs"]) if SPLIT_PROTOCOL == "spec_full" else
            "{v} source normals held out as validation (shared by A and B); early stopping patience {p}; checkpoint = best validation epoch.\n".format(v=len(data.val_idx), p=COMMON_CONFIG["early_stop_patience"])))
    for v in cks: (D["models"] / machine / VARIANT_DIR[v] / "TRAINING_PROTOCOL.txt").write_text(note)
    # optional cross-fitted calibration scores (independent normal scores for Part 2 threshold selection)
    if CALIBRATION_FOLDS and CALIBRATION_FOLDS >= 2: calibration_scores(machine, data)
    # 6. Mahalanobis statistics (Experiment 2 preparation)
    reps = {v: mahalanobis_statistics(machine, v, cks[v], data) for v in cks}
    mark(STATE, machine, "mahalanobis", all(r["all_checks_ok"] for r in reps.values()))
    # 7. verification
    ver = [verify_model(machine, v, mdir, data) for v in cks]
    vdf = pd.DataFrame(ver); vdf.to_csv(D["verify"] / f"{machine}_verification.csv", index=False)
    lines = [f"# Verification report - {machine}", f"Generated {datetime.datetime.now().isoformat()} | smoke_test={SMOKE_TEST}", "", "## Dataset audit", f"- files audited: {summary['files_audited']} (all files: {summary['audited_all_files']})",
             f"- train source normal {summary['train_source_normal']} (expected {EXPECTED['train_source_normal']}), train target normal {summary['train_target_normal']} (expected {EXPECTED['train_target_normal']})",
             f"- test source {summary['test_source']} = {summary['test_source_normal']} normal + {summary['test_source_anomaly']} anomaly; test target {summary['test_target']} = {summary['test_target_normal']} normal + {summary['test_target_anomaly']} anomaly",
             f"- sample rates {summary['sample_rates']}, channels {summary['channels']}, duration {summary['duration_min_s']}..{summary['duration_max_s']} s, unreadable {summary['unreadable']}, duplicate names {summary['duplicate_filenames']}, duplicate contents {summary['duplicate_contents']}",
             f"- problems: {summary['problems'] or 'none'}", "", f"## Split (protocol {SPLIT_PROTOCOL})",
             f"- recordings available: {len(data.src_files)} source normal, {len(data.tgt_files)} target normal | reserved for validation: {len(data.val_idx)} source | used for gradient updates: Model A {len(data.tr_idx)} source, Model B {len(data.tr_idx)} source + {len(data.tgt_files)} target",
             "", "## Models"]
    for v in cks:
        ck = cks[v]; lines.append(f"- {v}: epochs run {ck['epochs_run']}, selected epoch {ck['best_epoch']} ({'final epoch' if math.isnan(ck['val_loss']) else 'best validation'}), train loss {ck['train_loss']:.5f}" + ("" if math.isnan(ck['val_loss']) else f", val loss {ck['val_loss']:.5f}") + f", {ck['total_seconds']/60:.1f} min, device {ck['device']}" + ("  (reused from existing ToyCar run)" if machine == "ToyCar" and TOYCAR_REUSE.get("compatible") else ""))
    lines += ["", "## Mahalanobis statistics"] + [f"- {v}: groups {list(reps[v]['groups'])}, numerically valid {reps[v]['all_checks_ok']}" for v in reps]
    lines += ["", "## Verification checks", vdf.T.to_string(), "", f"## RESULT: {'ALL PASSED' if vdf.passed.all() else 'FAILED - see table'}"]
    (D["verify"] / f"{machine}_verification.md").write_text("\n".join(lines), encoding="utf-8")
    passed = bool(vdf.passed.all()) and all(r["all_checks_ok"] for r in reps.values())
    req = required_files_for(machine); absent = [str(p.relative_to(PROJECT_ROOT)) for p in req if not p.exists()]
    if absent: log(f"  {machine}: required outputs missing on Drive: {absent}"); passed = False
    mark(STATE, machine, "required_files_present", not absent)
    mark(STATE, machine, "verification", passed, status="COMPLETE" if passed else "VERIFICATION_FAILED")
    log(f"=== {machine}: {'COMPLETE' if passed else 'VERIFICATION FAILED'} ===")
    # 8. cleanup local copies only after everything is on Drive
    if passed and DELETE_LOCAL_AFTER_MACHINE:
        shutil.rmtree(local, ignore_errors=True)
        if not FEATURE_CACHE_ON_DRIVE: shutil.rmtree(cache_dir, ignore_errors=True)
        log(f"  local files for {machine} removed ({disk_info(LOCAL_WORK)})")
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

for machine in MACHINE_ORDER:
    try:
        process_machine(machine)
    except Exception as e:
        log(f"!!! {machine}: ERROR {type(e).__name__}: {e}"); traceback.print_exc()
        mark(STATE, machine, "error", False, note=f"{type(e).__name__}: {str(e)[:300]}", status="ERROR")
print("\nPipeline state:", json.dumps({m: STATE[m]["status"] for m in MACHINE_ORDER}, indent=1))

## Section 7 — Stage 4: cross-machine summary
Combined dataset audit files, one training summary row per model (14 rows when all seven machines are complete), a grid of loss curves. As in the ToyCar notebook, A and B losses are on different normalisation scales and are described per model; no ranking is implied. The actual comparison happens in Part 2 on the test sets.

In [ ]:
# ---- 7.1 Combined audit, training summary, loss-curve grid -------------------------------------------
audits = [pd.read_csv(p) for p in sorted(D["audit"].glob("dataset_audit_*.csv"))]
if audits:
    ALL_AUDIT = pd.concat(audits, ignore_index=True); ALL_AUDIT.to_csv(D["audit"] / "dataset_audit.csv", index=False)
    ALL_AUDIT[["machine_type", "split_folder", "filename", "domain", "label", "attributes", "sr", "channels", "duration_s", "size_bytes", "md5"]].to_csv(D["audit"] / "dataset_file_manifest.csv", index=False)
summaries = {p.stem.replace("dataset_summary_", ""): json.loads(p.read_text()) for p in sorted(D["audit"].glob("dataset_summary_*.json"))}
(D["audit"] / "dataset_summary.json").write_text(json.dumps(dict(expected=EXPECTED, archives=ARCHIVE_INFO, machines=summaries, pipeline_state={m: STATE[m]["status"] for m in MACHINE_ORDER}), indent=2, default=str))
rep = ["# Dataset integrity report", f"Generated {datetime.datetime.now().isoformat()}", "", "| machine | archive | status | src train (exp 990) | tgt train (exp 10) | test src n/a (exp 100) | test tgt n/a (exp 100) | sr | ch | duration s | unreadable | dup names | dup contents | problems |", "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for m in MACHINE_ORDER:
    s = summaries.get(m); a = ARCHIVE_INFO[m]
    if s: rep.append(f"| {m} | {a['zip']} | {STATE[m]['status']} | {s['train_source_normal']} | {s['train_target_normal']} | {s['test_source_normal']}/{s['test_source_anomaly']} | {s['test_target_normal']}/{s['test_target_anomaly']} | {s['sample_rates']} | {s['channels']} | {s['duration_min_s']}..{s['duration_max_s']} | {s['unreadable']} | {s['duplicate_filenames']} | {s['duplicate_contents']} | {'; '.join(s['problems']) or 'none'} |")
    else: rep.append(f"| {m} | {a['zip']} | {STATE[m]['status']} | not audited | | | | | | | | | | {'archive missing' if not a['exists'] else 'not processed yet'} |")
rep += ["", "Supplemental (noise-only) clips present in the archives were not extracted and were not used. Test recordings were never used for training, validation or model selection."]
(D["audit"] / "dataset_integrity_report.md").write_text("\n".join(rep), encoding="utf-8")

rows = []
for m in MACHINE_ORDER:
    for v in ("source_only", "source_target"):
        p = D["models"] / m / VARIANT_DIR[v] / f"{v}_autoencoder.pth"; h = D["hist"] / f"{m}_{v}_training_history.csv"
        if p.exists() and h.exists():
            ck = torch.load(p, map_location="cpu", weights_only=False); hist = pd.read_csv(h)
            rows.append(dict(machine=m, model=v, status=STATE[m]["status"], source_recordings=len(ck["train_files"]["source"]), target_recordings=len(ck["train_files"]["target"]), validation_recordings=len(ck["validation_files"]),
                             train_windows=ck["n_train_windows"], epochs_run=ck.get("epochs_run"), selected_epoch=ck.get("best_epoch"), checkpoint_selection=ck.get("checkpoint_selection", ""),
                             best_val_loss=(hist.val_loss.min() if "val_loss" in hist else "n/a (no validation set)"), final_train_loss=hist.train_loss.iloc[-1],
                             training_minutes=round(ck.get("total_seconds", float("nan")) / 60, 1), device=ck["device"], reused=(m == "ToyCar" and TOYCAR_REUSE.get("compatible", False))))
TRAIN_SUMMARY = pd.DataFrame(rows); TRAIN_SUMMARY.to_csv(D["hist"] / "ALL_MACHINES_training_summary.csv", index=False); display(TRAIN_SUMMARY)
done_m = [m for m in MACHINE_ORDER if (D["hist"] / f"{m}_source_only_training_history.csv").exists() and (D["hist"] / f"{m}_source_target_training_history.csv").exists()]
if done_m:
    fig, axes = plt.subplots(len(done_m), 2, figsize=(13, 3.6 * len(done_m)), squeeze=False)
    for r, m in enumerate(done_m):
        for c, v in enumerate(("source_only", "source_target")):
            h = pd.read_csv(D["hist"] / f"{m}_{v}_training_history.csv"); ax = axes[r, c]
            ax.plot(h.epoch, h.train_loss, color="#2A6FDB", lw=1.8, label="training")
            if "val_loss" in h: ax.plot(h.epoch, h.val_loss, color="#E07B2A", lw=1.8, label="validation")
            ax.set_title(f"{m} - {v} (own normalisation scale)", loc="left", fontsize=10); ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=8)
            for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.tight_layout(); fig.savefig(D["curves"] / "ALL_MACHINES_loss_curves.png", dpi=120); plt.show(); plt.close(fig)
print("\nNOTE: losses are per-model normalised MSE; A vs B and machine vs machine values are not comparable rankings.")

## Section 8 — Stage 5: preprocessing code, verification summary, handover ZIP (tested)
Writes `dcase_part1_lib.py` (from the four `[LIB]` cells), the overall verification summary, the handover folder and `DCASE_2026_PART1_COMPLETE_HANDOVER.zip`, then re-opens the ZIP and checks that every required file is present for every complete machine.

In [ ]:
# ---- 8.1 Library module from the [LIB] cells + verification summary ------------------------------------
_lib = {}
for _src in get_ipython().user_ns.get("In", []):
    if _src.lstrip().startswith("# [LIB]"): _lib[_src.lstrip().splitlines()[0]] = _src
assert len(_lib) == 4, f"expected 4 [LIB] cells, found {list(_lib)}"
module = ["# dcase_part1_lib.py - generated by DCASE_2026_PART1_MASTER_COLAB.ipynb. Architecture, preprocessing, feature extraction, inference (all machines).",
          "import sys, json\nfrom pathlib import Path\nimport numpy as np\nimport librosa\nimport soundfile as sf\nimport torch\nfrom torch import nn\n"] + list(_lib.values()) + ['''
def load_mahalanobis_stats(npz_path):
    # returns dict with latent_mean/latent_precision (8-d) and residual_mean/residual_precision (frame_dim-d) plus the raw and regularised covariances
    z = np.load(npz_path); return {k: z[k] for k in z.files}

def mahalanobis_sq(x, mean, precision):
    # squared Mahalanobis distance of each row of x (n, d) to N(mean, Sigma) given Sigma^-1
    d = x - mean; return np.einsum("ni,ij,nj->n", d, precision, d)

def extract_latent_and_residual(wav_path, checkpoint_path):
    # convenience for Part 2: returns latent (n_windows, 8) and residual frames (n_windows*context, frame_dim) for one recording
    model, ck = load_checkpoint(checkpoint_path); x = wav_to_model_input(wav_path, ck); res = run_inference(model, x)
    return dict(latent=res["latent"], residual_frames=res["residual"].reshape(-1, ck["frame_dim"]), window_mse=res["window_mse"], config=ck["config"], frame_dim=ck["frame_dim"])
''']
(D["pre_code"] / "dcase_part1_lib.py").write_text("\n\n".join(module), encoding="utf-8")
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location("dcase_part1_lib", D["pre_code"] / "dcase_part1_lib.py"); _mod = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_mod)
print("dcase_part1_lib.py written and importable:", [n for n in dir(_mod) if not n.startswith("_")][:14], "...")

vers = [pd.read_csv(p) for p in sorted(D["verify"].glob("*_verification.csv"))]
VER_ALL = pd.concat(vers, ignore_index=True) if vers else pd.DataFrame()
if len(VER_ALL): VER_ALL.to_csv(D["verify"] / "ALL_MACHINES_verification_summary.csv", index=False); display(VER_ALL[["machine", "variant", "passed", "input_dim", "finite_on_real_audio", "reproducible", "mahalanobis_stats_valid"]])

In [ ]:
# ---- 8.2 Handover folder + ZIP + content test -------------------------------------------------------------
H = D["handover"]
if H.exists(): shutil.rmtree(H)
H.mkdir(parents=True)
complete = []
for m in MACHINE_ORDER:
    if STATE[m]["status"] != "COMPLETE": continue
    gone = [str(p.relative_to(PROJECT_ROOT)) for p in required_files_for(m) if not p.exists()]
    if gone:
        mark(STATE, m, "required_files_present", False, note=f"files missing on Drive at handover time: {gone[:5]}", status="INCOMPLETE_FILES_MISSING")
        log(f"{m}: downgraded from COMPLETE, required files missing: {gone[:5]} -> re-run the notebook to regenerate")
    else: complete.append(m)
for m in complete:
    for v in ("source_only", "source_target"):
        mdst = H / "Models" / m / VARIANT_DIR[v]; mdst.mkdir(parents=True, exist_ok=True)
        for fn in (f"{v}_autoencoder.pth", f"{v}_model_config.json", f"{v}_COMPLETE.json", "TRAINING_PROTOCOL.txt"): shutil.copy2(D["models"] / m / VARIANT_DIR[v] / fn, mdst / fn)
        shutil.copy2(D["pre_cfg"] / f"{m}_{v}_feature_norm_stats.npz", mdst / f"{v}_feature_norm_stats.npz")
        shutil.copy2(D["hist"] / f"{m}_{v}_training_history.csv", (H / "Training_Histories").mkdir(parents=True, exist_ok=True) or H / "Training_Histories" / f"{m}_{v}_training_history.csv")
        shutil.copy2(D["curves"] / f"{m}_{v}_loss_curve.png", (H / "Loss_Curves").mkdir(parents=True, exist_ok=True) or H / "Loss_Curves" / f"{m}_{v}_loss_curve.png")
    shutil.copy2(D["curves"] / f"{m}_training_curves_per_model.png", H / "Loss_Curves" / f"{m}_training_curves_per_model.png")
    (H / "Mahalanobis_Statistics" / m).mkdir(parents=True, exist_ok=True)
    for p in (D["mahala"] / m).glob("*"):
        if p.suffix in (".npz", ".json") and "COMPLETE" not in p.name: shutil.copy2(p, H / "Mahalanobis_Statistics" / m / p.name)
    for fn in (f"{m}_split_manifest.json", f"{m}_preprocessing_config.json", f"{m}_feature_verification.json"): shutil.copy2(D["pre_cfg"] / fn, (H / "Preprocessing").mkdir(parents=True, exist_ok=True) or H / "Preprocessing" / fn)
    for fn in (f"{m}_verification.md", f"{m}_verification.csv"): shutil.copy2(D["verify"] / fn, (H / "Verification_Reports").mkdir(parents=True, exist_ok=True) or H / "Verification_Reports" / fn)
    for fn in (f"dataset_summary_{m}.json", f"dataset_audit_{m}.csv"): shutil.copy2(D["audit"] / fn, (H / "Dataset_Audit").mkdir(parents=True, exist_ok=True) or H / "Dataset_Audit" / fn)
for fn in ("dataset_audit.csv", "dataset_summary.json", "dataset_file_manifest.csv", "dataset_integrity_report.md", "archive_structure.json"):
    if (D["audit"] / fn).exists(): (H / "Dataset_Audit").mkdir(parents=True, exist_ok=True); shutil.copy2(D["audit"] / fn, H / "Dataset_Audit" / fn)
(H / "Preprocessing").mkdir(parents=True, exist_ok=True); shutil.copy2(D["pre_code"] / "dcase_part1_lib.py", H / "Preprocessing" / "dcase_part1_lib.py")
if (D["models"] / "ToyCar" / "Previous_Holdout_Validation_Run").exists(): shutil.copytree(D["models"] / "ToyCar" / "Previous_Holdout_Validation_Run", H / "Models" / "ToyCar" / "Previous_Holdout_Validation_Run", dirs_exist_ok=True)
(H / "Configurations").mkdir(parents=True, exist_ok=True)
for fn in ("common_config.json", "environment.json", "pipeline_state.json"): shutil.copy2(D["cfg"] / fn, H / "Configurations" / fn)
for fn in ("ALL_MACHINES_training_summary.csv",):
    if (D["hist"] / fn).exists(): shutil.copy2(D["hist"] / fn, H / "Training_Histories" / fn) if (H / "Training_Histories").exists() else None
if (D["curves"] / "ALL_MACHINES_loss_curves.png").exists(): (H / "Loss_Curves").mkdir(parents=True, exist_ok=True); shutil.copy2(D["curves"] / "ALL_MACHINES_loss_curves.png", H / "Loss_Curves" / "ALL_MACHINES_loss_curves.png")
if (D["verify"] / "ALL_MACHINES_verification_summary.csv").exists(): (H / "Verification_Reports").mkdir(parents=True, exist_ok=True); shutil.copy2(D["verify"] / "ALL_MACHINES_verification_summary.csv", H / "Verification_Reports" / "ALL_MACHINES_verification_summary.csv")

# README + specification with the actual numbers of this run
tbl = ["| machine | status | model | gradient-update recordings | validation recordings (0 = none) | epochs | selected epoch | validation loss |", "|---|---|---|---|---|---|---|---|"]
for _, r in (TRAIN_SUMMARY.iterrows() if len(TRAIN_SUMMARY) else []):
    tbl.append(f"| {r.machine} | {r.status} | {r.model} | {r.source_recordings} src + {r.target_recordings} tgt | {r.validation_recordings} | {r.epochs_run} | {r.selected_epoch} | {r.best_val_loss if isinstance(r.best_val_loss, str) else f'{r.best_val_loss:.5f}'} |")
spec = f'''# PART 1 handover - all machine types ({"SMOKE TEST - not trained models" if SMOKE_TEST else "full training"})
Generated {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}. Machines COMPLETE: {complete}. Not complete: {[m for m in MACHINE_ORDER if m not in complete]} (see Configurations/pipeline_state.json).

## Training protocol: {SPLIT_PROTOCOL}
{"Model A: 990 source-domain normal recordings. Model B: 990 source + 10 target-domain normal recordings. Fixed 100 epochs. Final-epoch checkpoint. No independent validation set, no validation loss, no early stopping." if SPLIT_PROTOCOL == "spec_full" else "Model A: 891 source normals; Model B: 891 source + 10 target normals; 99 source normals held out as validation; early stopping (patience 15); best-validation checkpoint."} Details per machine: Preprocessing/<machine>_split_manifest.json and TRAINING_PROTOCOL.txt in each model folder.

## Models
{chr(10).join(tbl)}
ToyCar models reused from the earlier finished ToyCar run: {TOYCAR_REUSE.get("compatible", False)}.{"" if TOYCAR_REUSE.get("compatible", False) else " The earlier ToyCar run used the holdout protocol and is included unchanged as Models/ToyCar/Previous_Holdout_Validation_Run/ for reference only; the spec_full ToyCar models in Models/ToyCar/Source_Only and Source_Target were trained separately."}

## Layout
Models/<machine>/Source_Only|Source_Target/: <variant>_autoencoder.pth ({"final-epoch checkpoint of the fixed 100-epoch budget" if SPLIT_PROTOCOL == "spec_full" else "best-validation checkpoint"}), <variant>_model_config.json, <variant>_feature_norm_stats.npz, <variant>_COMPLETE.json, TRAINING_PROTOCOL.txt
Mahalanobis_Statistics/<machine>/: <variant>_<group>_mahalanobis_stats.npz for group in source_train, target_train, all_train{", validation_source" if SPLIT_PROTOCOL != "spec_full" else ""}; <variant>_mahalanobis_report.json
Training_Histories/, Loss_Curves/, Preprocessing/ (dcase_part1_lib.py, per-machine split manifests and configs), Dataset_Audit/, Verification_Reports/, Configurations/

## Interface (identical for every machine)
Input: WAV -> dcase_part1_lib.wav_to_model_input(path, ck) -> (n_windows, {INPUT_DIM}) float32, z-scored with the checkpoint's own norm_mean/norm_std.
Model: DCASEAutoencoder({INPUT_DIM}) -> (reconstruction (n_windows, {INPUT_DIM}), latent (n_windows, 8)).
Residual frames: (input - reconstruction).reshape(-1, {FRAME_DIM}).
    import sys; sys.path.append("<handover>/Preprocessing"); import dcase_part1_lib as p1
    model, ck = p1.load_checkpoint("<handover>/Models/Fan/Source_Only/source_only_autoencoder.pth")
    x = p1.wav_to_model_input("<fan>/test/section_00_source_test_normal_0000_<attr>.wav", ck); res = p1.run_inference(model, x)
    S = p1.load_mahalanobis_stats("<handover>/Mahalanobis_Statistics/Fan/source_only_source_train_mahalanobis_stats.npz")
    d_latent = p1.mahalanobis_sq(res["latent"], S["latent_mean"], S["latent_precision"])                              # per window, 8-d latent
    d_resid  = p1.mahalanobis_sq(res["residual"].reshape(-1, ck["frame_dim"]), S["residual_mean"], S["residual_precision"])  # per frame, {FRAME_DIM}-d residual
MSE (Experiment 1) = res["window_mse"] (reconstruction error). Mahalanobis (Experiment 2) = statistical distance to the normal distribution; the two are different scores. How to aggregate per clip, which group statistics to use (source_train / target_train / all_train, e.g. selective min over domains) and all thresholds, AUC, pAUC, precision, recall, F1 are Part 2 decisions.

## Part 2 still has to do
Anomaly scoring (MSE, Mahalanobis latent and/or residual), threshold selection, AUC/pAUC, precision/recall/F1, comparison of Experiments 1, 2 and 3 across machines, application.
'''
(H / "README.md").write_text(spec, encoding="utf-8"); (H / "PART1_HANDOVER_SPECIFICATION.md").write_text(spec, encoding="utf-8")
if D["calib"].exists() and any(D["calib"].glob("*_out_of_fold_normal_scores.csv")):
    (H / "Calibration").mkdir(exist_ok=True)
    for p in D["calib"].glob("*.csv"): shutil.copy2(p, H / "Calibration" / p.name)
    for p in D["calib"].glob("*_calibration_COMPLETE.json"): shutil.copy2(p, H / "Calibration" / p.name)
protocol_md = f'''# Training protocol and threshold-calibration guidance (Part 1 -> Part 2)

## Protocol used: {SPLIT_PROTOCOL}
''' + ('''Model A: all 990 source-domain normal training recordings. Model B: all 990 source + 10 target-domain normal training recordings. Fixed budget of 100 epochs, Adam 1e-3, batch 256, seed 42.
**There is no independent held-out validation set.** No early stopping was performed, no validation loss exists, and the checkpoint is the final epoch (the official DCASE baseline follows the same protocol). The training-loss curves show optimisation progress only; they are not estimates of generalisation and must not be presented as validation results.''' if SPLIT_PROTOCOL == "spec_full" else
'''99 source normals per machine were held out once (seed 42) as a validation set shared by Model A and Model B; early stopping with patience 15; checkpoint = best validation epoch. Model A trained on 891 source, Model B on 891 source + 10 target normals.''') + f'''

## How Part 2 can calibrate decision thresholds without test recordings or test labels
Test recordings and their labels are reserved for evaluation only. The options below use training-split normals only; each is stated with its bias.

1. **Official-baseline method (training-score distribution).** Score the training normals (source_train for A; all_train for B) with the deliverable model, fit a gamma distribution to the per-clip scores and take a high quantile (the baseline uses the 90th percentile). Bias: the model has fitted these recordings, so their scores are optimistically low and the threshold tends to be too low (more false positives on unseen normals). State this when reporting. These are training recordings, not independent validation data.
2. **Cross-fitted (out-of-fold) normal scores - recommended when available.** Part 1 can produce `Calibration/<machine>_out_of_fold_normal_scores.csv` (set `CALIBRATION_FOLDS >= 2` in the master notebook): K fold models, each trained on (K-1)/K of the source normals, score the fold they never saw. The pooled out-of-fold scores of all 990 source normals are genuinely independent normal scores for that pipeline. Use their distribution (e.g. a quantile, or a gamma fit) to set the threshold for the deliverable model's scores of the same type (MSE, latent Mahalanobis, residual Mahalanobis). Bias: fold models saw ~{(CALIBRATION_FOLDS - 1) / CALIBRATION_FOLDS * 100 if CALIBRATION_FOLDS else 80:.0f} % of the data and {CALIBRATION_EPOCHS} epochs, so their score scale can differ slightly from the full model; apply the threshold as a quantile rank or rescale by matching the median of training scores between fold models and the full model. Status in this handover: {"included" if (H / "Calibration").exists() else "NOT generated in this run (CALIBRATION_FOLDS = 0)"}.
3. **Target-domain normals for Model A only.** The 10 target training normals were not used to train Model A; their scores under Model A are independent but few (10) and domain-shifted. Useful as a sanity check for the target domain, not as the sole calibration set. For Model B they are training recordings.
4. **Do not** fit thresholds on test recordings, choose thresholds that maximise F1 on test labels, or use the held-out ToyCar validation recordings of the earlier run (`Previous_Holdout_Validation_Run`) for the spec_full models (different models).

Whatever option is used, the report should name the calibration set, its size, and the bias above. Threshold-free metrics (AUC, pAUC) do not depend on this choice.
'''
(H / "TRAINING_PROTOCOL_AND_CALIBRATION.md").write_text(protocol_md, encoding="utf-8")
(PROJECT_ROOT / "TRAINING_PROTOCOL_AND_CALIBRATION.md").write_text(protocol_md, encoding="utf-8")
zip_path = PROJECT_ROOT / ("DCASE_2026_PART1_COMPLETE_HANDOVER_SMOKE.zip" if SMOKE_TEST else "DCASE_2026_PART1_COMPLETE_HANDOVER.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(H.rglob("*")):
        if p.is_file(): z.write(p, arcname=str(p.relative_to(H.parent)))
# test the zip
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None, "zip corrupted"; names = set(z.namelist())
required = [f"Handover/Preprocessing/dcase_part1_lib.py", "Handover/README.md", "Handover/PART1_HANDOVER_SPECIFICATION.md", "Handover/Dataset_Audit/dataset_integrity_report.md", "Handover/Configurations/common_config.json", "Handover/TRAINING_PROTOCOL_AND_CALIBRATION.md"]
for m in complete:
    for v in ("source_only", "source_target"):
        required += [f"Handover/Models/{m}/{VARIANT_DIR[v]}/{v}_autoencoder.pth", f"Handover/Models/{m}/{VARIANT_DIR[v]}/{v}_model_config.json", f"Handover/Models/{m}/{VARIANT_DIR[v]}/{v}_feature_norm_stats.npz",
                     f"Handover/Models/{m}/{VARIANT_DIR[v]}/{v}_COMPLETE.json", f"Handover/Models/{m}/{VARIANT_DIR[v]}/TRAINING_PROTOCOL.txt", f"Handover/Training_Histories/{m}_{v}_training_history.csv", f"Handover/Loss_Curves/{m}_{v}_loss_curve.png",
                     f"Handover/Mahalanobis_Statistics/{m}/{v}_mahalanobis_report.json"] + [f"Handover/Mahalanobis_Statistics/{m}/{v}_{g}_mahalanobis_stats.npz" for g in ("source_train", "target_train", "all_train")]
    required += [f"Handover/Verification_Reports/{m}_verification.md", f"Handover/Verification_Reports/{m}_verification.csv", f"Handover/Preprocessing/{m}_split_manifest.json",
                 f"Handover/Preprocessing/{m}_preprocessing_config.json", f"Handover/Preprocessing/{m}_feature_verification.json", f"Handover/Dataset_Audit/dataset_audit_{m}.csv",
                 f"Handover/Dataset_Audit/dataset_summary_{m}.json", f"Handover/Loss_Curves/{m}_training_curves_per_model.png"]
absent = [r for r in required if r not in names]
print(f"HANDOVER ZIP: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB, {len(names)} files) | machines included: {complete}")
for n in sorted(names)[:400]: print("  ", n)
if absent: print("MISSING IN ZIP:", absent)
else: log(f"HANDOVER ZIP VERIFIED: all {len(required)} required files present for {len(complete)} machine(s): {zip_path}")
mark(STATE, "_handover", "handover_zip", not absent, note=f"zip with {complete}", status="VERIFIED" if not absent else "INCOMPLETE")

## Section 9 — Stage 6: implementation report and final status
Generates `PART1_IMPLEMENTATION_REPORT.md` from the real outputs of this run (nothing is typed in by hand) and prints the final per-machine status. Machines that are not `COMPLETE` are listed with the reason and the exact next action.

In [ ]:
# ---- 9.1 Implementation report with actual numbers ----------------------------------------------------------
def md_table(df, cols):
    if not len(df): return "(none yet)"
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows(): out.append("| " + " | ".join(f"{r[c]:.5f}" if isinstance(r[c], float) else str(r[c]) for c in cols) + " |")
    return "\n".join(out)
status_lines = []
for m in MACHINE_ORDER:
    s = STATE[m]; nxt = {"COMPLETE": "nothing", "MISSING": f"upload {ARCHIVES[m]} to {DRIVE_DATA_DIR} and re-run", "INCOMPLETE": "inspect Dataset_Audit/dataset_summary_<m>.json; fix archive or set STRICT_COUNTS=False",
                         "ERROR": "see logs/part1_master.log, fix, re-run (resumes)", "INCOMPLETE_FILES_MISSING": "re-run the notebook; missing outputs are regenerated", "in_progress": "re-run the notebook (resumes)", "pending": "re-run the notebook", "VERIFICATION_FAILED": "see Verification_Reports"}.get(s["status"], "re-run")
    status_lines.append(f"| {m} | {s['status']} | {'; '.join(s['notes'][-2:]) if s['notes'] else ''} | {nxt} |")
report = f'''# PART 1 implementation report - DCASE 2026 Task 2 ({"SMOKE TEST RUN" if SMOKE_TEST else "full run"})
Generated {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | device {DEVICE} | {VERSIONS}

## 1. Project overview
Noise-aware unsupervised anomalous sound detection (DCASE 2026 Task 2). Part 1 prepares, for each machine type, the dataset audit, two-channel log-Mel features, two trained Autoencoders (Experiment 1 / 3: Model A source-only, Model B source + target) and the normal-distribution statistics needed for Experiment 2 (Mahalanobis). Part 2 (teammate) performs the anomaly scoring and evaluation.

## 2. Dataset organisation (observed, not assumed)
Archives expected in {DRIVE_DATA_DIR}: {list(ARCHIVES.values())}.
Per machine the project document expects {EXPECTED}. Observed counts, sample rates, channels, durations, duplicates and problems: see Dataset_Audit/dataset_integrity_report.md (summary below).
{(D["audit"] / "dataset_integrity_report.md").read_text().split(chr(10), 3)[-1] if (D["audit"] / "dataset_integrity_report.md").exists() else ""}

## 3. Preprocessing methodology
soundfile load (both channels, float32) -> resample to 16 kHz only if needed -> per channel Mel power spectrogram (n_fft 1024, hop 512, 128 mels, power 2) -> 20/power*log10(max(mel, eps)) -> channels concatenated (256-dim frame; Channel 1 = index 0, Channel 2 = index 1, neutral labels) -> nan_to_num -> per-dimension z-score with statistics of the model's own training recordings -> 5 consecutive frames flattened (1280-dim). Durations are not padded or cropped (frame-based model). Features are cached per machine ({"on Drive" if FEATURE_CACHE_ON_DRIVE else "on Colab local disk"}). Supplemental noise clips are not used.
Deviation from the official baseline: the baseline reads both channels (mono: false) but uses only the first for features and applies no normalisation; this custom pipeline uses both channels and z-scoring. Results are therefore not comparable to the official baseline numbers.

## 4. Autoencoder
DCASEAutoencoder: 1280 -> 128 -> 128 -> 128 -> 128 -> 8 -> 128 -> 128 -> 128 -> 128 -> 1280, BatchNorm1d + ReLU after hidden layers, Linear output; 432 408 parameters; MSE loss; Adam lr 1e-3; batch 256; seed 42; {"fixed 100 epochs, final-epoch checkpoint, no validation set, no early stopping" if SPLIT_PROTOCOL == "spec_full" else "up to 100 epochs, early stopping (patience 15) on the held-out validation set, best-validation checkpoint"}. One separate model per machine and per experiment variant (14 models for seven machines).

## 5. Experiments
Split protocol used in this run: **{SPLIT_PROTOCOL}**. {"Model A uses ALL 990 source normals and Model B ALL 990 source + 10 target normals for gradient updates, fixed 100 epochs, final-epoch checkpoint; no recording is held out, so there is no validation loss (the AI_LAB.pdf / official-baseline protocol). Overfitting cannot be measured without a held-out set; training-loss convergence is monitored instead." if SPLIT_PROTOCOL == "spec_full" else "99 of the 990 source normals are held out (seed 42) as a validation set shared by A and B; Model A trains on 891 source, Model B on 891 source + 10 target; early stopping (patience 15) and best-validation checkpoint. The nominal 990 vs 1000 comparison is realised as 891 vs 901 gradient-update recordings."} Exact counts per machine are recorded in Preprocessing/<m>_split_manifest.json and in each checkpoint. Test recordings were never used for training, validation or model selection.
Experiment 2 (Mahalanobis) preparation: for each trained model, mean, covariance and regularised precision (Sigma + lambda I, lambda = 1e-3 trace/d, Cholesky-solved inverse) of (a) the 8-dim latent vectors and (b) the 256-dim residual frames, over the groups source_train (990 source normals), target_train (10 target normals), all_train (1000){" and validation_source (99 held-out normals)" if SPLIT_PROTOCOL != "spec_full" else ""}; all groups contain only normal recordings from train/. Dimensions, finiteness, positive definiteness, condition numbers and inverse accuracy are verified per group. MSE and Mahalanobis are kept as distinct scores; Part 2 chooses aggregation, group and thresholds.

## 6. Training results actually produced in this run
{md_table(TRAIN_SUMMARY, ["machine", "model", "status", "source_recordings", "target_recordings", "validation_recordings", "epochs_run", "selected_epoch", "checkpoint_selection", "final_train_loss", "training_minutes", "reused"]) if len(TRAIN_SUMMARY) else "(no model trained yet)"}
Losses are per-model normalised MSE and are not comparable between A and B or between machines. Lower loss means better reconstruction of normal sound, not better anomaly detection.
ToyCar: {"reused from the earlier finished run (" + str({k: TOYCAR_REUSE[k] for k in ("source_only", "source_target") if k in TOYCAR_REUSE}) + ")" if TOYCAR_REUSE.get("compatible") else "trained in this run or not yet processed"}.

## 7. Verification
{(VER_ALL[["machine", "variant", "passed", "finite_on_real_audio", "reproducible", "norm_stats_loadable", "mahalanobis_stats_valid"]].to_markdown(index=False) if len(VER_ALL) and hasattr(VER_ALL, "to_markdown") else (VER_ALL.to_string() if len(VER_ALL) else "(none yet)"))}

## 8. Status per machine
| machine | status | notes | next action |
|---|---|---|---|
{chr(10).join(status_lines)}

## 9. Limitations
1. Only machines with status COMPLETE have trained models and statistics; nothing is reported for the others.
2. The experiments use the custom two-channel, normalised pipeline; no claim of reproducing the official baseline.
3. {"spec_full: no independent validation set exists, so no early stopping was possible and generalisation cannot be estimated from training loss; the final-epoch checkpoint of the fixed 100-epoch budget is the model. Threshold calibration for Part 2 must rely on training-split normals or on the optional cross-fitted calibration scores (see TRAINING_PROTOCOL_AND_CALIBRATION.md)." if SPLIT_PROTOCOL == "spec_full" else "holdout_10pct: the validation hold-out reduces the training sets to 891 / 901 recordings, so the nominal 990 / 1000 comparison is approximated."}
4. Anomaly detection quality is unknown until Part 2 evaluates on the test sets.
{"5. Cross-fitted calibration scores were generated with CALIBRATION_FOLDS=" + str(CALIBRATION_FOLDS) + " fold models (" + str(CALIBRATION_EPOCHS) + " epochs each); fold models are not the deliverable models and their score scale may differ slightly from the full models." if CALIBRATION_FOLDS and CALIBRATION_FOLDS >= 2 else "5. No cross-fitted calibration scores were generated in this run (CALIBRATION_FOLDS=0)."}
'''
(PROJECT_ROOT / "PART1_IMPLEMENTATION_REPORT.md").write_text(report, encoding="utf-8")
shutil.copy2(PROJECT_ROOT / "PART1_IMPLEMENTATION_REPORT.md", D["handover"] / "PART1_IMPLEMENTATION_REPORT.md")
print(report[:6000])
print("\n==== FINAL STATUS ====")
for m in MACHINE_ORDER: print(f"  {m:14s} {STATE[m]['status']}")
log("=== session end ===")